# Bus ETA study — additional October data
The target remains at least 80% of test predictions with an absolute error no greater than 30 seconds. The data include the previous June study and two exports dated 27 October 2026. All four required model families are retained.

The October GPS export has 29 vehicles and 500 records per vehicle. The arrival export covers unit 16 only. The analysis uses GPS U16 with logbook unit 16, supported by matching checkpoint passages. This correspondence is not provider-verified. Speed uses the previous km/h convention; its unit is not separately confirmed for the new export. No clock shift is applied.

Only arrivals covered by GPS and existing control zones can be used. October trips are partially observed. June and October results are identified separately. The future-day test trains only on June; the additional trip-based study can train on both dates and is an offline comparison.

In Colab, place the five original files from input/ in /content, or set DATA_DIR_OVERRIDE. Run cells in order. The notebook contains all study code. The full search trains many models and can take a long time.
The current success target is 80% within the unchanged ±30-second tolerance. The previous version used an 85% target. This update changes the stated success target, not the measured errors or test observations.


In [1]:
from pathlib import Path
import os, sys, subprocess, importlib.util
try:
    IN_COLAB = importlib.util.find_spec('google.colab') is not None
except ModuleNotFoundError:
    IN_COLAB = False
if IN_COLAB:
    loaded_versions = {name:getattr(sys.modules.get(name),'__version__',None) for name in ['numpy','pandas']}
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
        'pandas==3.0.6', 'numpy==2.5.3', 'scipy==1.18.1', 'scikit-learn==1.9.1',
        'shapely==2.2.0', 'xgboost==3.4.1', 'lightgbm==4.7.0',
        'plotly==7.1.0', 'matplotlib==3.11.2', 'openpyxl==3.1.5'])
    expected_core={'numpy':'2.5.3','pandas':'3.0.6'}
    if any(v is not None and v!=expected_core[name] for name,v in loaded_versions.items()):
        raise RuntimeError('Dependencies installed. In Colab choose Runtime > Restart session, then Run all again. This prevents mixing old loaded libraries with newly installed versions.')
import json, re, hashlib, unicodedata, platform, warnings, time, copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from shapely.geometry import Point, Polygon, LineString, mapping
from shapely.ops import transform
from shapely import covers, points
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from threadpoolctl import threadpool_limits
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from IPython.display import display, HTML
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment
import plotly.graph_objects as go
from plotly.offline import get_plotlyjs

DATA_DIR_OVERRIDE = ''
OUTPUT_DIR_OVERRIDE = ''
MATCH_TOLERANCE_SEC = 120
PRIMARY_TOLERANCE_SEC = 30
SUCCESS_TARGET_PCT = 80
HORIZON_MIN_SEC, HORIZON_MAX_SEC = 30, 300
VEHICLE_MAP = {'r4-u7': '1','r4_de_puebla-u16':'16'}  # Correspondence used in this study; not checked with the data provider.
SPEED_LIMIT_KMH = None        # Set only after obtaining a documented applicable limit.
SENSITIVITY_SEEDS = [11, 23, 37, 51, 79]
TRAINING_PERCENTAGES = list(range(100, 0, -10))
N_JOBS = 2
threadpool_limits(limits=N_JOBS)  # Avoid BLAS/OpenMP oversubscription on small datasets.
SEED = 42
np.random.seed(SEED)
plt.rcParams.update({'figure.dpi':130, 'savefig.dpi':200, 'font.size':10,
                     'axes.spines.top':False, 'axes.spines.right':False})

def find_data_dir():
    candidates = [Path(DATA_DIR_OVERRIDE)] if DATA_DIR_OVERRIDE else [Path.cwd()/'input', Path.cwd(), Path.cwd().parent, Path('/content'), Path('/content/drive/MyDrive/Data')]
    for d in candidates:
        if d.exists() and list(d.glob('GPS_record_Ux*.xlsx')) and list(d.glob('Bitacora*.xlsx')) and list(d.glob('route_description*.geojson')):
            return d.resolve()
    raise FileNotFoundError('Upload GPS_record_Ux, Bitacora and route_description, or set DATA_DIR_OVERRIDE.')
DATA_DIR = find_data_dir()
OUT = Path(OUTPUT_DIR_OVERRIDE) if OUTPUT_DIR_OVERRIDE else Path.cwd() / 'outputs'
for sub in ['data', 'tables', 'figures', 'dashboard', 'documentation']:
    (OUT/sub).mkdir(parents=True, exist_ok=True)
GPS_PATH = sorted(DATA_DIR.glob('GPS_record_Ux*.xlsx'))[0]
LOG_PATH = sorted(DATA_DIR.glob('Bitacora*.xlsx'))[0]
GEO_PATH = sorted(DATA_DIR.glob('route_description*.geojson'))[0]
def sha256(p): return hashlib.sha256(p.read_bytes()).hexdigest()
source_hashes = {p.name:sha256(p) for p in DATA_DIR.iterdir() if p.is_file() and p.suffix.lower() in ['.xlsx','.geojson','.csv']}
environment = {m.__name__:m.__version__ for m in [np,pd,openpyxl]}
for name in ['scikit-learn','shapely','xgboost','lightgbm','plotly','matplotlib','scipy']:
    import importlib.metadata
    environment[name]=importlib.metadata.version(name)
(OUT/'documentation'/'environment.json').write_text(json.dumps({'python':platform.python_version(), 'packages':environment},indent=2))
print('Data directory:', DATA_DIR)
print('Primary criterion: absolute error <=30 s; target >=80%.')

Data directory: C:\Users\LENOVO\Desktop\Hanine\work_update_20261028\input
Primary criterion: absolute error <=30 s; target >=85%.


## 1. Source audit and consistent labels
Original input files remain unchanged. Old Excel latitude/longitude labels are reversed; the new CSV coordinates are correctly named. Each source is therefore renamed separately. English CSV copies preserve the source field values. GPS records for buses without arrival logs remain in the source audit, but cannot become labelled training examples.

In [2]:
import csv
gps_labels={'Unit_id':'vehicle_id','Velocidad':'speed_kmh','timestap':'timestamp','Latitud':'longitude','Longitud':'latitude'}
log_labels={'Fecha':'date','Unidad':'logbook_vehicle_id','Vuelta':'trip_id','Punto de chequeo':'checkpoint','Llegada ideal':'scheduled_arrival_time','Llegada real':'actual_arrival_time'}
new_gps_labels={'Unit_id':'vehicle_id','Velocidad':'speed_kmh','Timestamp':'timestamp','Latitud':'latitude','Longitud':'longitude'}
new_log_labels={'date':'date','unit_number':'logbook_vehicle_id','path_number':'trip_id','checkpoint_name':'checkpoint','checkpoint_street_address':'street_address','checkpoint_arrival_time':'scheduled_elapsed_time','to_char':'scheduled_arrival_time','to_char-2':'actual_arrival_time','delay_time_tolerance':'delay_time','advance_time_tolerance':'advance_time'}
NEW_GPS=DATA_DIR/'data-1793151398974.csv';NEW_LOG=DATA_DIR/'data-1793151897219-dashboard.csv'
assert NEW_GPS.exists() and NEW_LOG.exists()
for source,labels,name in [(NEW_GPS,new_gps_labels,'October_GPS_English.csv'),(NEW_LOG,new_log_labels,'October_logbook_English.csv')]:
    with source.open(encoding='utf-8-sig',newline='') as f:
        reader=csv.reader(f);header=next(reader);values=list(reader)
    target=OUT/'data'/name
    with target.open('w',encoding='utf-8',newline='') as f:
        writer=csv.writer(f);writer.writerow([labels.get(c,c) for c in header]);writer.writerows(values)
    with target.open(encoding='utf-8',newline='') as f:assert list(csv.reader(f))[1:]==values
old_gps=pd.read_excel(GPS_PATH).rename(columns=gps_labels)
old_log=pd.read_excel(LOG_PATH).rename(columns=log_labels)
all_new_gps=pd.read_csv(NEW_GPS).rename(columns=new_gps_labels)
new_log=pd.read_csv(NEW_LOG).rename(columns=new_log_labels)
vehicle_audit=all_new_gps.groupby('vehicle_id').agg(rows=('timestamp','size'),first_time=('timestamp','min'),last_time=('timestamp','max')).reset_index()
vehicle_audit['arrival_log_available']=vehicle_audit.vehicle_id.eq('R4_DE_PUEBLA-U16')
vehicle_audit.to_csv(OUT/'tables/October_vehicle_coverage.csv',index=False)
new_gps=all_new_gps[all_new_gps.vehicle_id=='R4_DE_PUEBLA-U16'].copy()
new_gps.vehicle_id=new_gps.vehicle_id.str.lower()
assert set(new_log.logbook_vehicle_id.astype(str))=={'16'}
new_log['source_trip_number']=new_log.trip_id
new_log['trip_id']=100+new_log.trip_id.astype(int)
old_log['source_trip_number']=old_log.trip_id
gps_raw=pd.concat([old_gps,new_gps],ignore_index=True)
log_raw=pd.concat([old_log,new_log],ignore_index=True)
print('New GPS:',len(all_new_gps),'rows,',all_new_gps.vehicle_id.nunique(),'vehicles; usable vehicle GPS:',len(new_gps))
display(vehicle_audit)

New GPS: 14500 rows, 29 vehicles; usable vehicle GPS: 500


,vehicle_id,rows,first_time,last_time,arrival_log_available
0,R4_DE_PUEBLA-U050,500,2026-10-27 17:36:46,2026-10-27 19:36:07,False
1,R4_DE_PUEBLA-U066,500,2026-10-27 18:12:49,2026-10-27 19:36:09,False
2,R4_DE_PUEBLA-U1,500,2026-10-27 18:12:48,2026-10-27 19:36:05,False
3,R4_DE_PUEBLA-U10,500,2026-10-27 17:37:44,2026-10-27 19:35:59,False
4,R4_DE_PUEBLA-U100,500,2026-10-27 16:44:49,2026-10-27 19:36:00,False
5,R4_DE_PUEBLA-U16,500,2026-10-27 16:57:45,2026-10-27 19:35:54,True
6,R4_DE_PUEBLA-U21,500,2026-10-27 18:11:08,2026-10-27 19:36:04,False
7,R4_DE_PUEBLA-U22,500,2026-10-27 18:12:54,2026-10-27 19:36:08,False
8,R4_DE_PUEBLA-U23,500,2026-10-27 16:52:34,2026-10-27 19:35:59,False
9,R4_DE_PUEBLA-U24,500,2026-10-27 17:03:56,2026-10-27 19:35:56,False


## 2. Cleaning, route matching and ETA labels
The same past-only features and one-to-one arrival matching are used on both dates. Original trip numbers are retained in the processed logbook; October test-group numbers are offset by 100 to avoid collisions with June groups. These identifiers are not predictors.

The 120-second matching rule and 30–300-second ETA study window stay unchanged. June groups cover complete available trips. October groups cover the recorded parts of trips 7 and 8; they are not full additional journeys.

In [3]:
gps=gps_raw.copy()
gps['source_row']=np.arange(2,len(gps)+2)
gps['timestamp']=pd.to_datetime(gps['timestamp'],errors='coerce')
for c in ['latitude','longitude','speed_kmh']: gps[c]=pd.to_numeric(gps[c],errors='coerce')
audit=[]
def remove_rows(frame,mask,reason):
    audit.append({'step':reason,'removed_rows':int(mask.sum()),'remaining_rows':int((~mask).sum())})
    return frame.loc[~mask].copy()
gps=remove_rows(gps,gps[['vehicle_id','timestamp','latitude','longitude','speed_kmh']].isna().any(axis=1),'Missing essential GPS field')
gps=remove_rows(gps,gps.duplicated(['vehicle_id','timestamp','latitude','longitude','speed_kmh']),'Exact duplicate GPS record')
gps=remove_rows(gps,(gps.speed_kmh<0)|~gps.latitude.between(-90,90)|~gps.longitude.between(-180,180),'Invalid speed or coordinate')
conflicts=gps.duplicated(['vehicle_id','timestamp'],keep=False)
gps=remove_rows(gps,conflicts,'Conflicting records at same vehicle timestamp')
gps=gps.sort_values(['vehicle_id','timestamp']).reset_index(drop=True)
gps['gps_row_id']=np.arange(len(gps))
gps['sampling_gap_sec']=gps.groupby('vehicle_id').timestamp.diff().dt.total_seconds()
gps['logbook_vehicle_id']=gps.vehicle_id.map(VEHICLE_MAP)
assert gps.logbook_vehicle_id.notna().all(), 'Explicit vehicle mapping required'
assert (gps.latitude.between(18,20)&gps.longitude.between(-100,-97)).all(), 'Coordinate meaning inconsistent with supplied route'
log=log_raw.copy()
log['logbook_row_id']=np.arange(len(log))
log['logbook_vehicle_id']=log.logbook_vehicle_id.astype(str)
log['trip_id']=pd.to_numeric(log.trip_id,errors='raise').astype(int)
def normalize_name(x):
    s=unicodedata.normalize('NFKD',str(x).strip().upper())
    return re.sub(r'\s+',' ',''.join(c for c in s if not unicodedata.combining(c)))
log['checkpoint_key']=log.checkpoint.map(normalize_name)
log['actual_arrival']=pd.to_datetime(log.date.astype(str)+' '+log.actual_arrival_time.astype(str),errors='coerce',format='mixed')
log['scheduled_arrival']=pd.to_datetime(log.date.astype(str)+' '+log.scheduled_arrival_time.astype(str),errors='coerce',format='mixed')
for _,g in log.groupby(['date','logbook_vehicle_id','trip_id'],sort=False):
    previous=None
    for i in g.index:
        t=log.loc[i,'scheduled_arrival']
        if previous is not None and pd.notna(t):
            while t<previous: t+=pd.Timedelta(days=1)
        log.loc[i,'scheduled_arrival']=t
        previous=t if pd.notna(t) else previous
log['arrival_status']=np.where(log.actual_arrival.notna(),'Recorded','No recorded arrival')
pd.DataFrame(audit).to_csv(OUT/'tables'/'gps_cleaning_audit.csv',index=False)
log.to_csv(OUT/'data'/'logbook_processed.csv',index=False)
quality=pd.DataFrame([
 {'measure':'Raw GPS rows','value':len(gps_raw)}, {'measure':'Clean GPS rows','value':len(gps)},
 {'measure':'GPS vehicles','value':gps.vehicle_id.nunique()}, {'measure':'GPS days','value':gps.timestamp.dt.date.nunique()},
 {'measure':'Raw logbook rows','value':len(log)}, {'measure':'Missing actual arrivals','value':log.actual_arrival.isna().sum()},
 {'measure':'Median GPS interval (s)','value':gps.sampling_gap_sec.median()},
 {'measure':'Maximum GPS interval (s)','value':gps.sampling_gap_sec.max()},
 {'measure':'Maximum source speed (km/h)','value':gps.speed_kmh.max()}])
quality.to_csv(OUT/'tables'/'source_quality.csv',index=False)
display(quality)
display(pd.DataFrame(audit))

,measure,value
0,Raw GPS rows,9140.00
1,Clean GPS rows,8398.00
2,GPS vehicles,2.00
3,GPS days,2.00
4,Raw logbook rows,250.00
5,Missing actual arrivals,18.00
6,Median GPS interval (s),10.00
7,Maximum GPS interval (s),73.00
8,Maximum source speed (km/h),42.03


,step,removed_rows,remaining_rows
0,Missing essential GPS field,0,9140
1,Exact duplicate GPS record,742,8398
2,Invalid speed or coordinate,0,8398
3,Conflicting records at same vehicle timestamp,0,8398


In [4]:
geo_text=GEO_PATH.read_text(encoding='utf-8')
repaired_text=re.sub(r',\s*([}\]])',r'\1',geo_text)
geo=json.loads(repaired_text)
zone_polys={}; geometry_audit=[]; route_coords=None
for i,f in enumerate(geo['features']):
    props=f.get('properties') or {}; coords=(f.get('geometry') or {}).get('coordinates')
    if props.get('recorrido_nombre') and coords: route_coords=np.asarray(coords,float)
    if not props.get('nombre') or not coords: continue
    name=normalize_name(props['nombre']); poly=Polygon([(lon,lat) for lat,lon in coords])
    if not poly.is_valid or poly.area<=0: raise ValueError('Invalid control-zone polygon: '+name)
    duplicate=name in zone_polys
    if duplicate: assert zone_polys[name].equals(poly), 'Conflicting checkpoint geometries: '+name
    else: zone_polys[name]=poly
    geometry_audit.append({'feature_index':i,'checkpoint':name,'valid':True,'duplicate':duplicate})
assert route_coords is not None
LAT0=float(route_coords[:,0].mean()); LON0=float(route_coords[:,1].mean())
M_PER_DEG=6371000*np.pi/180; COSLAT=np.cos(np.deg2rad(LAT0))
def project_xy(lon,lat): return ((np.asarray(lon)-LON0)*M_PER_DEG*COSLAT,(np.asarray(lat)-LAT0)*M_PER_DEG)
zone_metric={k:transform(project_xy,p) for k,p in zone_polys.items()}
for r in geometry_audit:
    p=zone_metric[r['checkpoint']]
    r['approximate_area_m2']=p.area
    r['approximate_perimeter_m']=p.length
    r['zone_size_source']='Unmodified supplied polygon; not tuned on outcomes'
route_xy=np.column_stack(project_xy(route_coords[:,1],route_coords[:,0]))
route_line=LineString(route_xy)
seg_a=route_xy[:-1]; seg_v=np.diff(route_xy,axis=0)
seg_len=np.linalg.norm(seg_v,axis=1); seg_len2=np.maximum(seg_len**2,1e-12)
seg_s=np.r_[0,np.cumsum(seg_len)]; ROUTE_LENGTH=float(seg_len.sum())
def project_stream(frame):
    xx,yy=project_xy(frame.longitude.values,frame.latitude.values)
    positions=[]; offsets=[]; previous=None; previous_t=None
    for x,y,t in zip(xx,yy,frame.timestamp):
        point=np.array([x,y]); u=np.clip(np.sum((point-seg_a)*seg_v,axis=1)/seg_len2,0,1)
        distance=np.linalg.norm(point-(seg_a+u[:,None]*seg_v),axis=1)
        s=seg_s[:-1]+u*seg_len
        score=distance.copy()
        plausible=distance<=distance.min()+20.0
        if previous is not None:
            dt=max((t-previous_t).total_seconds(),1)
            advance=(s-previous+ROUTE_LENGTH/2)%ROUTE_LENGTH-ROUTE_LENGTH/2
            score+=np.maximum(-advance-35,0)*0.8+np.maximum(advance-(45*dt+100),0)*0.8
        score[~plausible]=np.inf
        j=int(np.argmin(score))
        assert distance[j]<=distance.min()+20.0+1e-8
        positions.append(s[j]); offsets.append(distance[j]); previous=s[j]; previous_t=t
    return np.array(positions),np.array(offsets)
gps['route_progress_m']=np.nan; gps['route_offset_m']=np.nan
for _,g in gps.groupby('vehicle_id'):
    s,d=project_stream(g)
    gps.loc[g.index,'route_progress_m']=s; gps.loc[g.index,'route_offset_m']=d
# Route samples identify fixed first-entry locations for each control zone.
sample_s=np.arange(0,ROUTE_LENGTH,5.0)
sample_pts=[route_line.interpolate(float(s)) for s in sample_s]
zone_entries={}
for name,p in zone_metric.items():
    inside=np.array([p.covers(pt) for pt in sample_pts])
    entering=inside & ~np.r_[False,inside[:-1]]
    entries=sample_s[entering]
    if not len(entries): entries=np.array([route_line.project(p.centroid)])
    zone_entries[name]=entries
repaired_geo={'type':'FeatureCollection','features':[{'type':'Feature','properties':{'feature_type':'route'},'geometry':mapping(LineString([(lon,lat) for lat,lon in route_coords]))}]+[
    {'type':'Feature','properties':{'checkpoint':k},'geometry':mapping(p)} for k,p in zone_polys.items()]}
(OUT/'data'/'route_description_repaired.geojson').write_text(json.dumps(repaired_geo,indent=2))
pd.DataFrame(geometry_audit).to_csv(OUT/'tables'/'geometry_audit.csv',index=False)
print('Distinct control zones:',len(zone_polys),'| Projected route length (km):',round(ROUTE_LENGTH/1000,2))
print('Checkpoint names without a zone:',sorted(set(log.checkpoint_key)-set(zone_polys)))
route_quality=pd.DataFrame([{'metric':'mean_offset_m','value':gps.route_offset_m.mean()},
    {'metric':'median_offset_m','value':gps.route_offset_m.median()},
    {'metric':'p90_offset_m','value':gps.route_offset_m.quantile(.9)},
    {'metric':'max_offset_m','value':gps.route_offset_m.max()},
    {'metric':'gps_fixes_offset_above_1000m','value':int((gps.route_offset_m>1000).sum())}])
route_quality.to_csv(OUT/'tables'/'route_matching_quality.csv',index=False)

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_22876\769500302.py:19: DeprecationWarning: The 'shapely.ops.transform()' function is deprecated. Use 'shapely.transform()' or 'shapely.transform_coordseq()' instead. See https://shapely.readthedocs.io/en/latest/manual.html#shapely.ops.transform for more details.
  zone_metric={k:transform(project_xy,p) for k,p in zone_polys.items()}


Distinct control zones: 12 | Projected route length (km): 32.87
Checkpoint names without a zone: ['135 PTE', '25 PONIENTE', '39 PONIENTE', 'BASE CENTRO', 'LAGUNA', 'PUENTE SAN BARTOLO', 'TORRES R']


In [5]:
trip_times=log[log.actual_arrival.notna()].groupby(['logbook_vehicle_id','trip_id']).actual_arrival.agg(['min','max']).reset_index()
trip_bounds=[]
for unit,g in trip_times.groupby('logbook_vehicle_id'):
    g=g.sort_values('trip_id').reset_index(drop=True)
    # Trips with at least two recorded arrivals; the incomplete eighth trip is excluded.
    counts=log[log.actual_arrival.notna()].groupby(['logbook_vehicle_id','trip_id']).size()
    g=g[g.trip_id.map(lambda t:counts.loc[(unit,t)])>=2].reset_index(drop=True)
    for i,r in g.iterrows():
        start=r['min'] if i==0 else g.loc[i-1,'max']+(r['min']-g.loc[i-1,'max'])/2
        # Include the final arrival and its GPS confirmation margin.
        end=r['max']+pd.Timedelta(seconds=MATCH_TOLERANCE_SEC,nanoseconds=1) if i==len(g)-1 else r['max']+(g.loc[i+1,'min']-r['max'])/2
        trip_bounds.append({'logbook_vehicle_id':unit,'trip_id':int(r.trip_id),'start':start,'end':end})
bounds=pd.DataFrame(trip_bounds)
gps['trip_id']=pd.Series(pd.NA,index=gps.index,dtype='Int64')
for r in bounds.itertuples():
    mask=(gps.logbook_vehicle_id==r.logbook_vehicle_id)&(gps.timestamp>=r.start)&(gps.timestamp<r.end)
    gps.loc[mask,'trip_id']=r.trip_id
stream=gps[gps.trip_id.notna()].copy()
stream['trip_id']=stream.trip_id.astype(int)
events=[]
for (unit,trip),g in stream.groupby(['vehicle_id','trip_id']):
    for name,p in zone_polys.items():
        inside=np.asarray(covers(p,points(g.longitude.values,g.latitude.values)))
        entering=inside & ~np.r_[False,inside[:-1]]
        for i in np.flatnonzero(entering):
            r=g.iloc[i]
            events.append({'vehicle_id':unit,'logbook_vehicle_id':r.logbook_vehicle_id,'trip_id':trip,
                'checkpoint':name,'gps_arrival':r.timestamp,'gps_row_id':int(r.gps_row_id),
                'latitude':r.latitude,'longitude':r.longitude,'speed_kmh':r.speed_kmh,
                'route_progress_m':r.route_progress_m,'entry_has_previous_outside_fix':bool(i>0),
                'previous_fix_time':g.iloc[i-1].timestamp if i>0 else pd.NaT,
                'entry_sampling_interval_sec':(r.timestamp-g.iloc[i-1].timestamp).total_seconds() if i>0 else np.nan})
events=pd.DataFrame(events).sort_values(['vehicle_id','trip_id','gps_arrival']).reset_index(drop=True)
events['gps_event_id']=np.arange(len(events))
def match_events(tolerance):
    used=set(); rows=[]; exclusions=[]
    for r in log.sort_values(['actual_arrival','logbook_row_id'],na_position='last').itertuples():
        reason=None
        if pd.isna(r.actual_arrival): reason='No recorded actual arrival'
        elif r.trip_id not in set(bounds.trip_id): reason='Incomplete trip excluded'
        elif r.checkpoint_key not in zone_polys: reason='No supplied control-zone geometry'
        if reason:
            exclusions.append({'logbook_row_id':r.logbook_row_id,'trip_id':r.trip_id,'checkpoint':r.checkpoint,'reason':reason}); continue
        c=events[(events.logbook_vehicle_id==r.logbook_vehicle_id)&(events.trip_id==r.trip_id)&
            (events.checkpoint==r.checkpoint_key)&(events.gps_arrival.dt.date==r.actual_arrival.date())&~events.gps_event_id.isin(used)].copy()
        if len(c):
            c['signed_diff_sec']=(c.gps_arrival-r.actual_arrival).dt.total_seconds()
            c['abs_diff_sec']=c.signed_diff_sec.abs()
            c=c[c.abs_diff_sec<=tolerance].sort_values(['abs_diff_sec','gps_arrival','gps_event_id'])
        if c.empty:
            exclusions.append({'logbook_row_id':r.logbook_row_id,'trip_id':r.trip_id,'checkpoint':r.checkpoint,'reason':'No eligible GPS passage within tolerance'});continue
        best=c.iloc[0]; used.add(int(best.gps_event_id))
        rows.append({**best.to_dict(),'logbook_row_id':r.logbook_row_id,'actual_arrival':r.actual_arrival,
            'scheduled_arrival':r.scheduled_arrival,'candidate_count':len(c),
            'event_id':f'{r.actual_arrival.date()}|{best.vehicle_id}|{r.trip_id}|{r.checkpoint_key}|{r.actual_arrival.isoformat()}'})
    return pd.DataFrame(rows),pd.DataFrame(exclusions)
matched,excluded=match_events(MATCH_TOLERANCE_SEC)
assert matched.gps_event_id.is_unique and matched.logbook_row_id.is_unique and matched.event_id.is_unique
matched=matched.sort_values(['vehicle_id','trip_id','actual_arrival']).reset_index(drop=True)
delta=matched.signed_diff_sec
delta_stats=pd.DataFrame([{'matched_events':len(matched),'mean_signed_sec':delta.mean(),'mean_absolute_sec':delta.abs().mean(),
 'median_absolute_sec':delta.abs().median(),'sd_signed_sec':delta.std(),'p90_absolute_sec':delta.abs().quantile(.9),'max_absolute_sec':delta.abs().max()}])
threshold_matching=pd.DataFrame([{'matching_tolerance_sec':t,'matched_events':len(match_events(t)[0])} for t in [0,10,30,60,120]])
events.to_csv(OUT/'data'/'gps_geofence_events.csv',index=False)
matched.to_csv(OUT/'data'/'matched_arrival_events.csv',index=False)
excluded.to_csv(OUT/'tables'/'logbook_exclusions.csv',index=False)
delta_stats.to_csv(OUT/'tables'/'arrival_matching_statistics.csv',index=False)
threshold_matching.to_csv(OUT/'tables'/'matching_tolerance_sensitivity.csv',index=False)
display(delta_stats);display(threshold_matching)
print('Detected GPS passages:',len(events),'| Matched:',len(matched),'| Excluded logbook rows:',len(excluded))

,matched_events,mean_signed_sec,mean_absolute_sec,median_absolute_sec,sd_signed_sec,p90_absolute_sec,max_absolute_sec
0,88,0.0,0.0,0.0,0.0,0.0,0.0


,matching_tolerance_sec,matched_events
0,0,88
1,10,88
2,30,88
3,60,88
4,120,88


Detected GPS passages: 130 | Matched: 88 | Excluded logbook rows: 162


In [6]:
def causal_features(g):
    g=g.sort_values('timestamp').copy().set_index('timestamp')
    for sec in [30,60,120]:
        g[f'speed_mean_{sec}s']=g.speed_kmh.rolling(f'{sec}s',closed='right').mean()
        g[f'speed_std_{sec}s']=g.speed_kmh.rolling(f'{sec}s',closed='right').std().fillna(0)
    g['stopped_fraction_60s']=(g.speed_kmh<1).astype(float).rolling('60s',closed='right').mean()
    dt=g.index.to_series().diff().dt.total_seconds()
    g['acceleration_mps2']=(g.speed_kmh.diff()/3.6/dt).fillna(0)
    x,y=project_xy(g.longitude.values,g.latitude.values)
    g['x_m']=x;g['y_m']=y
    g['dx_m']=g.x_m.diff().fillna(0);g['dy_m']=g.y_m.diff().fillna(0)
    g['position_speed_kmh']=(np.hypot(g.dx_m,g.dy_m)/dt*3.6).fillna(0)
    g['heading_cos']=g.dx_m/np.maximum(np.hypot(g.dx_m,g.dy_m),1)
    g['heading_sin']=g.dy_m/np.maximum(np.hypot(g.dx_m,g.dy_m),1)
    return g.reset_index()
stream=pd.concat([causal_features(g) for _,g in gps.groupby('vehicle_id')],ignore_index=True)
samples=[]
for r in matched.itertuples():
    w=stream[(stream.vehicle_id==r.vehicle_id)&(stream.trip_id==r.trip_id)&
        (stream.timestamp>=r.actual_arrival-pd.Timedelta(seconds=HORIZON_MAX_SEC))&
        (stream.timestamp<r.actual_arrival-pd.Timedelta(seconds=HORIZON_MIN_SEC))].copy()
    if w.empty: continue
    zone=zone_metric[r.checkpoint]; cent=zone.centroid
    # Use the same GPS-to-zone-centre distance as the reference study.
    cp=zone_polys[r.checkpoint].centroid
    lat1=np.radians(w.latitude.values);lon1=np.radians(w.longitude.values)
    lat2=np.radians(cp.y);lon2=np.radians(cp.x)
    a=np.sin((lat2-lat1)/2)**2+np.cos(lat1)*np.cos(lat2)*np.sin((lon2-lon1)/2)**2
    w['distance_centroid_km']=6371*2*np.arcsin(np.sqrt(np.clip(a,0,1)))
    w['distance_boundary_m']=[zone.distance(Point(x,y)) for x,y in zip(w.x_m,w.y_m)]
    distances=(zone_entries[r.checkpoint][None,:]-w.route_progress_m.values[:,None])%ROUTE_LENGTH
    w['route_remaining_m']=distances.min(axis=1)
    # A point inside the zone has zero remaining route distance.
    inside=np.asarray(covers(zone_polys[r.checkpoint],points(w.longitude.values,w.latitude.values)))
    w.loc[inside,'route_remaining_m']=0
    to_x=cent.x-w.x_m;to_y=cent.y-w.y_m; norm=np.maximum(np.hypot(to_x,to_y),1)
    w['towards_checkpoint_cos']=(to_x*w.heading_cos+to_y*w.heading_sin)/norm
    w['speed_distance_eta_sec']=w.distance_centroid_km/np.maximum(w.speed_mean_60s,3)*3600
    w['route_speed_eta_sec']=w.route_remaining_m/np.maximum(w.speed_mean_60s/3.6,3/3.6)
    w['target_eta_sec']=(r.actual_arrival-w.timestamp).dt.total_seconds()
    w['checkpoint']=r.checkpoint;w['event_id']=r.event_id;w['actual_arrival']=r.actual_arrival
    samples.append(w)
eta=pd.concat(samples,ignore_index=True).sort_values(['vehicle_id','trip_id','event_id','timestamp']).reset_index(drop=True)
seconds=eta.timestamp.dt.hour*3600+eta.timestamp.dt.minute*60+eta.timestamp.dt.second
eta['hour_sin']=np.sin(2*np.pi*seconds/86400);eta['hour_cos']=np.cos(2*np.pi*seconds/86400)
eta['instance_id']=np.arange(len(eta))
ORIGINAL_FEATURES=['speed_kmh','latitude','longitude','distance_centroid_km','speed_mean_30s','speed_mean_60s','speed_std_60s','hour_sin','hour_cos']
ENHANCED_FEATURES=ORIGINAL_FEATURES+['distance_boundary_m','route_remaining_m','route_progress_m','route_offset_m',
    'speed_mean_120s','speed_std_30s','speed_std_120s','stopped_fraction_60s','acceleration_mps2',
    'position_speed_kmh','heading_cos','heading_sin','towards_checkpoint_cos','sampling_gap_sec',
    'speed_distance_eta_sec','route_speed_eta_sec']
FEATURE_SETS={'Original':ORIGINAL_FEATURES,'Enhanced':ENHANCED_FEATURES}
CATEGORICAL_FEATURES=['checkpoint']
trips=sorted(eta.trip_id.unique())
assert len(trips)>=7 and eta.target_eta_sec.between(30,300,inclusive='right').all()
eta.to_csv(OUT/'data'/'eta_dataset.csv',index=False)
gps.to_csv(OUT/'data'/'gps_cleaned.csv',index=False)
cohort=eta.groupby('trip_id').agg(prediction_instances=('instance_id','size'),arrival_events=('event_id','nunique'))
cohort.to_csv(OUT/'tables'/'evaluation_cohort.csv')
display(cohort)
print('ETA rows:',len(eta),'| Distinct arrival events:',eta.event_id.nunique())

,prediction_instances,arrival_events
trip_id,,
1,297,11
2,296,11
3,272,11
4,297,11
5,297,11
6,277,11
7,168,11
107,15,1
108,147,10


ETA rows: 2066 | Distinct arrival events: 88


In [7]:
def advanced_motion(g):
    g=g.sort_values('timestamp').copy().set_index('timestamp')
    dt=g.index.to_series().diff().dt.total_seconds().fillna(1).clip(lower=1)
    stopped=g.speed_kmh<1
    stop_start=None;durations=[]
    for t,stop in zip(g.index,stopped):
        if not stop:stop_start=None
        elif stop_start is None:stop_start=t
        durations.append((t-stop_start).total_seconds() if stop_start is not None else 0.)
    g['stopped_duration_sec']=durations
    for sec in [60,120,180]:
        win=g.speed_kmh.rolling(f'{sec}s',closed='right')
        g[f'speed_median_{sec}s']=win.median()
        g[f'speed_max_{sec}s']=win.max()
    g['distance_travelled_60s_m']=(np.hypot(g.dx_m,g.dy_m)).rolling('60s',closed='right').sum()
    columns=['gps_row_id','stopped_duration_sec','distance_travelled_60s_m']+[f'speed_{stat}_{sec}s' for sec in [60,120,180] for stat in ['median','max']]
    result=[]
    for cp,zone in zone_metric.items():
        f=g[columns].copy()
        d=pd.Series([zone.distance(Point(x,y)) for x,y in zip(g.x_m,g.y_m)],index=g.index)
        # Measure change using current and earlier records only.
        closing=((d.shift(1)-d)/dt).fillna(0)
        f['distance_closing_30s']=closing.rolling('30s',closed='right').mean()
        f['distance_closing_60s']=closing.rolling('60s',closed='right').mean()
        f['checkpoint_dx_m']=zone.centroid.x-g.x_m
        f['checkpoint_dy_m']=zone.centroid.y-g.y_m
        f['approach_speed_kmh']=g.speed_kmh*(f.checkpoint_dx_m*g.heading_cos+f.checkpoint_dy_m*g.heading_sin)/np.maximum(np.hypot(f.checkpoint_dx_m,f.checkpoint_dy_m),1)
        f['closing_distance_eta_sec']=d/np.maximum(f.distance_closing_60s,1)
        f['checkpoint']=cp
        result.append(f.reset_index())
    return pd.concat(result,ignore_index=True)
advanced=pd.concat([advanced_motion(g) for _,g in stream.groupby('vehicle_id')],ignore_index=True)
ADVANCED_FEATURES=[c for c in advanced if c not in ['timestamp','gps_row_id','checkpoint']]
assert not advanced.duplicated(['gps_row_id','checkpoint']).any()
eta=eta.merge(advanced.drop(columns='timestamp'),on=['gps_row_id','checkpoint'],how='left',validate='many_to_one')
for cp in sorted(zone_polys):
    for feature in ['distance_centroid_km','speed_mean_60s','distance_closing_60s','stopped_duration_sec']:
        name='checkpoint_slope_'+cp.replace(' ','_')+'_'+feature
        eta[name]=eta[feature]*(eta.checkpoint==cp).astype(float)
        ADVANCED_FEATURES.append(name)
ENHANCED_FEATURES=ENHANCED_FEATURES+ADVANCED_FEATURES
FEATURE_SETS={'Original':ORIGINAL_FEATURES,'Enhanced':ENHANCED_FEATURES,
    'EnhancedNoClock':[c for c in ENHANCED_FEATURES if c not in ['hour_sin','hour_cos']],
    'MotionGeometry':[c for c in ENHANCED_FEATURES if c not in ['hour_sin','hour_cos','route_remaining_m','route_progress_m','route_offset_m','route_speed_eta_sec']]}
eta.to_csv(OUT/'data'/'eta_dataset.csv',index=False)
pd.DataFrame([{'feature_set':k,'feature':c} for k,v in FEATURE_SETS.items() for c in v]).to_csv(OUT/'documentation'/'model_feature_sets.csv',index=False)
# Changing future records must not change current features.
for trip in trips:
    g=stream[stream.vehicle_id==eta.loc[eta.trip_id==trip,'vehicle_id'].iloc[0]].sort_values('timestamp').reset_index(drop=True)
    ii=g.index[g.trip_id==trip].to_numpy()
    k=int(ii[len(ii)//2]);t=g.timestamp.iloc[k]
    raw=g.copy();changed=g.copy()
    changed.loc[k+1:,'speed_kmh']=55.
    changed.loc[k+1:,'x_m']+=5000;changed.loc[k+1:,'y_m']-=5000
    changed.loc[k+1:,'dx_m']+=100;changed.loc[k+1:,'dy_m']+=100
    a=advanced_motion(raw);b=advanced_motion(changed)
    cols=[c for c in a if c not in ['timestamp','gps_row_id','checkpoint']]
    assert np.allclose(a[a.timestamp<=t][cols],b[b.timestamp<=t][cols],equal_nan=True)
print('Additional feature future-perturbation checks passed on all available trip groups.')

Additional feature future-perturbation checks passed on all available trip groups.


In [8]:
checks=[]
rng=np.random.default_rng(42)
for _,g in stream.groupby(['vehicle_id','trip_id']):
    g=g.sort_values('timestamp').reset_index(drop=True)
    assert g.timestamp.is_monotonic_increasing and not g.timestamp.duplicated().any()
    for k in rng.choice(np.arange(5,len(g)-5),size=min(5,len(g)-10),replace=False):
        r=g.iloc[k]
        for sec in [30,60,120]:
            w=stream[(stream.vehicle_id==r.vehicle_id)&(stream.timestamp>r.timestamp-pd.Timedelta(seconds=sec))&(stream.timestamp<=r.timestamp)]
            assert np.isclose(w.speed_kmh.mean(),r[f'speed_mean_{sec}s'])
        raw=g[['vehicle_id','trip_id','timestamp','speed_kmh','longitude','latitude']].copy()
        pert=raw.copy();pert.loc[k+1:,'speed_kmh']=rng.uniform(0,60,len(pert)-k-1)
        for c in ['longitude','latitude']: pert.loc[k+1:,c]+=rng.normal(0,.0001,len(pert)-k-1)
        aa=causal_features(raw).iloc[k];bb=causal_features(pert).iloc[k]
        cols=[c for c in aa.index if c.startswith('speed_') or c in ['dx_m','dy_m','heading_cos','heading_sin','acceleration_mps2']]
        assert np.allclose(aa[cols].astype(float),bb[cols].astype(float))
        assert np.isclose(project_stream(raw)[0][k],project_stream(pert)[0][k])
    checks.append({'check':'Past-only rolling and map-matching future perturbation','trip_id':int(g.trip_id.iloc[0]),'passed':True})
for t in trips:
    a=eta[eta.trip_id!=t];b=eta[eta.trip_id==t]
    assert not set(a.event_id)&set(b.event_id)
    checks.append({'check':'Train/test arrival-event isolation','trip_id':int(t),'passed':True})
for _,r in matched.sample(min(8,len(matched)),random_state=42).iterrows():
    current=gps[gps.gps_row_id==r.gps_row_id].iloc[0]
    assert zone_polys[r.checkpoint].covers(Point(current.longitude,current.latitude))
assert not set(ENHANCED_FEATURES+CATEGORICAL_FEATURES)&{'target_eta_sec','actual_arrival','scheduled_arrival','event_id','trip_id','instance_id'}
pd.DataFrame(checks).to_csv(OUT/'documentation'/'leakage_checks.csv',index=False)
print('All',len(checks),'causality/partition checks passed.')

All 18 causality/partition checks passed.


## 3. Check the new evaluation data
The June feature rows must match the previous study. New records must have a matching recorded arrival and remain in a separate trip group. Missing arrivals, uncovered times and missing zone geometry are reported, not silently filled in.

In [9]:
eta['study_date']=eta.timestamp.dt.strftime('%Y-%m-%d')
eta['study_period']=np.where(eta.study_date=='2026-06-27','June','October')
eta.to_csv(OUT/'data/eta_dataset.csv',index=False)
coverage=eta.groupby(['study_date','vehicle_id','trip_id']).agg(prediction_instances=('instance_id','size'),arrival_events=('event_id','nunique'),first_prediction=('timestamp','min'),last_prediction=('timestamp','max')).reset_index()
coverage['trip_coverage']=np.where(coverage.study_date=='2026-06-27','Original trip coverage','Partial GPS export')
coverage.to_csv(OUT/'tables/evaluation_coverage.csv',index=False)
assert len(eta[eta.study_period=='June'])==1904 and eta[eta.study_period=='June'].event_id.nunique()==77
assert matched[matched.vehicle_id=='r4_de_puebla-u16'].event_id.nunique()==11
print('Evaluation:',len(eta),'observations,',eta.event_id.nunique(),'arrivals,',len(trips),'trip groups')
display(coverage)

Evaluation: 2066 observations, 88 arrivals, 9 trip groups


,study_date,vehicle_id,trip_id,prediction_instances,arrival_events,first_prediction,last_prediction,trip_coverage
0,2026-06-27,r4-u7,1,297,11,2026-06-27 07:01:13,2026-06-27 09:00:41,Original trip coverage
1,2026-06-27,r4-u7,2,296,11,2026-06-27 09:20:20,2026-06-27 11:22:37,Original trip coverage
2,2026-06-27,r4-u7,3,272,11,2026-06-27 11:33:14,2026-06-27 13:28:05,Original trip coverage
3,2026-06-27,r4-u7,4,297,11,2026-06-27 13:42:49,2026-06-27 15:56:09,Original trip coverage
4,2026-06-27,r4-u7,5,297,11,2026-06-27 16:30:26,2026-06-27 18:30:36,Original trip coverage
5,2026-06-27,r4-u7,6,277,11,2026-06-27 19:11:44,2026-06-27 21:12:04,Original trip coverage
6,2026-06-27,r4-u7,7,168,11,2026-06-27 21:26:27,2026-06-27 23:25:21,Original trip coverage
7,2026-10-27,r4_de_puebla-u16,107,15,1,2026-10-27 17:23:47,2026-10-27 17:28:07,Partial GPS export
8,2026-10-27,r4_de_puebla-u16,108,147,10,2026-10-27 18:08:29,2026-10-27 19:18:53,Partial GPS export


## 4. Models and a later-day test
Before adding October labels to training, test settings fixed from the June work. Train on all June trips and predict all eligible October examples. There is no October-based feature, setting or policy selection in this experiment.

The fixed four-model comparison uses the same EnhancedNoClock features. A separate comparison changes only Random Forest to its interval-based prediction policy. The policy predicts the centre of the 60-second interval with the largest estimated probability from training leaf values; this targets the fixed ±30-second success rule. No test ETA is used to choose the prediction.

In [10]:
def metrics(y,p):
    y=np.asarray(y,float);p=np.asarray(p,float);e=p-y
    return {'MAE_sec':mean_absolute_error(y,p),'RMSE_sec':np.sqrt(mean_squared_error(y,p)),
        'R2':r2_score(y,p),'MAPE_pct':100*np.mean(np.abs(e)/y),
        **{f'within_{t}s_pct':100*np.mean(np.abs(e)<=t) for t in [5,10,15,30,60]},
        'bias_sec':e.mean(),'p90_absolute_sec':np.quantile(np.abs(e),.9)}
MODEL_NAMES=['Linear Regression','Random Forest','XGBoost','LightGBM']
def estimator(name,variant='Default'):
    if name=='Linear Regression': return LinearRegression()
    expanded={
      'Random Forest':{
       'Deep':{'n_estimators':400,'max_depth':None,'min_samples_leaf':1,'max_features':.8},
       'Smooth':{'n_estimators':300,'max_depth':10,'min_samples_leaf':5,'max_features':1.0},
       'Subsample':{'n_estimators':400,'max_depth':None,'min_samples_leaf':2,'max_features':.5},
       'Absolute':{'n_estimators':120,'max_depth':10,'min_samples_leaf':5,'criterion':'absolute_error'}},
      'XGBoost':{
       'Shallow':{'n_estimators':450,'max_depth':2,'learning_rate':.04,'min_child_weight':3,'reg_lambda':5},
       'Depth4':{'n_estimators':450,'max_depth':4,'learning_rate':.04,'min_child_weight':3},
       'Penalized':{'n_estimators':450,'max_depth':3,'learning_rate':.05,'min_child_weight':8,'reg_lambda':15},
       'Absolute':{'n_estimators':350,'max_depth':3,'learning_rate':.05,'objective':'reg:absoluteerror'},
       'Dense':{'n_estimators':450,'max_depth':5,'learning_rate':.05,'subsample':1.,'colsample_bytree':1.}},
      'LightGBM':{
       'SmallLeaves':{'n_estimators':450,'num_leaves':7,'min_child_samples':10},
       'MediumLeaves':{'n_estimators':450,'num_leaves':15,'min_child_samples':8,'reg_lambda':5},
       'Flexible':{'n_estimators':450,'num_leaves':31,'min_child_samples':5,'reg_lambda':3},
       'Absolute':{'n_estimators':450,'num_leaves':15,'min_child_samples':10,'objective':'regression_l1'},
       'Regularized':{'n_estimators':450,'num_leaves':9,'min_child_samples':12}}}
    if variant in expanded[name]:
        model=estimator(name,'Default')
        return model.set_params(**expanded[name][variant])
    if name=='Random Forest':
        return RandomForestRegressor(n_estimators=300,max_depth=15 if variant=='Default' else 10,
            min_samples_leaf=2 if variant=='Default' else 5,random_state=SEED,n_jobs=N_JOBS)
    if name=='XGBoost':
        return XGBRegressor(n_estimators=300 if variant=='Default' else 450,max_depth=6 if variant=='Default' else 3,
            learning_rate=.05,subsample=.8,colsample_bytree=.8,objective='reg:squarederror',random_state=SEED,n_jobs=N_JOBS)
    return LGBMRegressor(n_estimators=300 if variant=='Default' else 450,learning_rate=.05,
        num_leaves=31 if variant=='Default' else 9,min_child_samples=20 if variant=='Default' else 12,
        random_state=SEED,n_jobs=N_JOBS,verbosity=-1,deterministic=True,force_col_wise=True)
def pipeline(name,feature_set='Enhanced',variant='Default'):
    prep=ColumnTransformer([
        ('numeric',Pipeline([('impute',SimpleImputer(strategy='median')),('scale',StandardScaler())]),FEATURE_SETS[feature_set]),
        ('categorical',OneHotEncoder(handle_unknown='ignore',sparse_output=False),CATEGORICAL_FEATURES)])
    return Pipeline([('preprocessing',prep),('model',estimator(name,variant))])
def fit_predict(train,test,name,feature_set='Enhanced',variant='Default'):
    p=pipeline(name,feature_set,variant)
    p.fit(train[FEATURE_SETS[feature_set]+CATEGORICAL_FEATURES],train.target_eta_sec)
    pred=np.maximum(p.predict(test[FEATURE_SETS[feature_set]+CATEGORICAL_FEATURES]),0)
    return pred,p
def prediction_frame(test,pred,name,experiment,trip,feature_set,variant):
    f=test[['instance_id','event_id','checkpoint','timestamp','actual_arrival','target_eta_sec','trip_id']].copy()
    f['model']=name;f['experiment']=experiment;f['predicted_eta_sec']=pred
    f['feature_set']=feature_set;f['variant']=variant
    f['arrival_error_sec']=f.predicted_eta_sec-f.target_eta_sec
    f['absolute_error_sec']=f.arrival_error_sec.abs()
    f['predicted_arrival']=f.timestamp+pd.to_timedelta(f.predicted_eta_sec,unit='s')
    assert np.allclose((f.predicted_arrival-f.actual_arrival).dt.total_seconds(),f.arrival_error_sec,atol=1e-6)
    return f

In [11]:
def forest_tolerance_predict(fitted,train,query,feature_set):
    cols=FEATURE_SETS[feature_set]+CATEGORICAL_FEATURES
    forest=fitted.named_steps['model'];prep=fitted.named_steps['preprocessing']
    train_leaves=forest.apply(prep.transform(train[cols]))
    query_leaves=forest.apply(prep.transform(query[cols]))
    values,index=np.unique(train.target_eta_sec.to_numpy(),return_inverse=True)
    probability=np.zeros((len(query),len(values)))
    for k in range(train_leaves.shape[1]):
        leaves,inv=np.unique(train_leaves[:,k],return_inverse=True)
        hist=np.zeros((len(leaves),len(values)))
        np.add.at(hist,(inv,index),1.)
        hist/=hist.sum(axis=1,keepdims=True)
        loc=np.searchsorted(leaves,query_leaves[:,k])
        assert np.all(leaves[loc]==query_leaves[:,k])
        probability+=hist[loc]/train_leaves.shape[1]
    assert np.allclose(probability.sum(axis=1),1.)
    # Test intervals whose edges fall on training ETA values.
    centres=np.unique(np.maximum(np.r_[values,values-30.,values+30.],0.))
    left=np.searchsorted(values,centres-30.,side='left')
    right=np.searchsorted(values,centres+30.,side='right')
    cumulative=np.c_[np.zeros(len(query)),np.cumsum(probability,axis=1)]
    mass=cumulative[:,right]-cumulative[:,left]
    ordinary=np.maximum(fitted.predict(query[cols]),0.)
    predicted=[]
    for row in range(len(query)):
        possible=np.flatnonzero(np.isclose(mass[row],mass[row].max(),rtol=0,atol=1e-12))
        best=min(possible,key=lambda j:(np.dot(probability[row],np.abs(values-centres[j])),abs(centres[j]-ordinary[row]),centres[j]))
        predicted.append(centres[best])
    return np.asarray(predicted)

In [12]:
JUNE=eta[eta.study_period=='June'].copy();OCTOBER=eta[eta.study_period=='October'].copy()
assert JUNE.timestamp.max()<OCTOBER.timestamp.min()
FROZEN_VARIANTS={'Linear Regression':'Default','Random Forest':'Deep','XGBoost':'Penalized','LightGBM':'Absolute'}
results=[];predictions_list=[]
def record(test,pred,model,experiment,features,variant):
    pf=prediction_frame(test,pred,model,experiment,0,features,variant)
    pf['study_period']=test.study_period.to_numpy();pf['study_date']=test.study_date.to_numpy()
    predictions_list.append(pf)
    for trip,g in pf.groupby('trip_id'):
        results.append({'experiment':experiment,'model':model,'test_trip':int(trip),'features':features,'variant':variant,
            'prediction_instances':len(g),'arrival_events':g.event_id.nunique(),**metrics(g.target_eta_sec,g.predicted_eta_sec)})
for name in MODEL_NAMES:
    variant=FROZEN_VARIANTS[name]
    pred,fitted=fit_predict(JUNE,OCTOBER,name,'EnhancedNoClock',variant)
    record(OCTOBER,pred,name,'Later-day fixed','EnhancedNoClock',variant)
    if name=='Random Forest':pred=forest_tolerance_predict(fitted,JUNE,OCTOBER,'EnhancedNoClock')
    record(OCTOBER,pred,name,'Later-day tolerance forest','EnhancedNoClock',variant)
pd.DataFrame(results).to_csv(OUT/'tables/later_day_results_by_trip.csv',index=False)
pd.concat(predictions_list,ignore_index=True).to_csv(OUT/'data/later_day_predictions.csv',index=False)
for exp in ['Later-day fixed','Later-day tolerance forest']:
    print(exp)
    for name in MODEL_NAMES:
        pf=next(p for p in predictions_list if p.experiment.iloc[0]==exp and p.model.iloc[0]==name)
        print(name,metrics(pf.target_eta_sec,pf.predicted_eta_sec),flush=True)

Later-day fixed
Linear Regression {'MAE_sec': 51.950226826063435, 'RMSE_sec': np.float64(221.9855121001357), 'R2': -7.0082313745601805, 'MAPE_pct': np.float64(36.095738708315984), 'within_5s_pct': np.float64(10.493827160493826), 'within_10s_pct': np.float64(22.22222222222222), 'within_15s_pct': np.float64(33.95061728395062), 'within_30s_pct': np.float64(65.4320987654321), 'within_60s_pct': np.float64(87.03703703703704), 'bias_sec': np.float64(18.506218573290806), 'p90_absolute_sec': np.float64(65.18690624156459)}


Random Forest {'MAE_sec': 23.91063271604938, 'RMSE_sec': np.float64(31.179804527579808), 'R2': 0.8420083530002938, 'MAPE_pct': np.float64(18.06368424582216), 'within_5s_pct': np.float64(13.580246913580247), 'within_10s_pct': np.float64(31.48148148148148), 'within_15s_pct': np.float64(41.9753086419753), 'within_30s_pct': np.float64(70.98765432098766), 'within_60s_pct': np.float64(93.82716049382715), 'bias_sec': np.float64(-7.792608024691358), 'p90_absolute_sec': np.float64(51.09450000000001)}


XGBoost {'MAE_sec': 24.051986199838144, 'RMSE_sec': np.float64(31.63162473817719), 'R2': 0.8373963277154806, 'MAPE_pct': np.float64(18.629592168129275), 'within_5s_pct': np.float64(11.728395061728394), 'within_10s_pct': np.float64(23.456790123456788), 'within_15s_pct': np.float64(41.9753086419753), 'within_30s_pct': np.float64(71.60493827160494), 'within_60s_pct': np.float64(91.35802469135803), 'bias_sec': np.float64(-8.045596534823193), 'p90_absolute_sec': np.float64(53.89180908203125)}


LightGBM {'MAE_sec': 24.31100599160203, 'RMSE_sec': np.float64(32.391406511582396), 'R2': 0.8294911351497897, 'MAPE_pct': np.float64(19.00432335818985), 'within_5s_pct': np.float64(12.345679012345679), 'within_10s_pct': np.float64(29.629629629629626), 'within_15s_pct': np.float64(43.20987654320987), 'within_30s_pct': np.float64(71.60493827160494), 'within_60s_pct': np.float64(93.20987654320987), 'bias_sec': np.float64(-7.6562943260082506), 'p90_absolute_sec': np.float64(54.460364275159385)}


Later-day tolerance forest
Linear Regression {'MAE_sec': 51.950226826063435, 'RMSE_sec': np.float64(221.9855121001357), 'R2': -7.0082313745601805, 'MAPE_pct': np.float64(36.095738708315984), 'within_5s_pct': np.float64(10.493827160493826), 'within_10s_pct': np.float64(22.22222222222222), 'within_15s_pct': np.float64(33.95061728395062), 'within_30s_pct': np.float64(65.4320987654321), 'within_60s_pct': np.float64(87.03703703703704), 'bias_sec': np.float64(18.506218573290806), 'p90_absolute_sec': np.float64(65.18690624156459)}


Random Forest {'MAE_sec': 28.67283950617284, 'RMSE_sec': np.float64(41.065966203742406), 'R2': 0.7259363477865268, 'MAPE_pct': np.float64(20.022141074242676), 'within_5s_pct': np.float64(6.172839506172839), 'within_10s_pct': np.float64(32.098765432098766), 'within_15s_pct': np.float64(38.2716049382716), 'within_30s_pct': np.float64(72.8395061728395), 'within_60s_pct': np.float64(91.35802469135803), 'bias_sec': np.float64(-12.685185185185185), 'p90_absolute_sec': np.float64(59.00000000000006)}


XGBoost {'MAE_sec': 24.051986199838144, 'RMSE_sec': np.float64(31.63162473817719), 'R2': 0.8373963277154806, 'MAPE_pct': np.float64(18.629592168129275), 'within_5s_pct': np.float64(11.728395061728394), 'within_10s_pct': np.float64(23.456790123456788), 'within_15s_pct': np.float64(41.9753086419753), 'within_30s_pct': np.float64(71.60493827160494), 'within_60s_pct': np.float64(91.35802469135803), 'bias_sec': np.float64(-8.045596534823193), 'p90_absolute_sec': np.float64(53.89180908203125)}


LightGBM {'MAE_sec': 24.31100599160203, 'RMSE_sec': np.float64(32.391406511582396), 'R2': 0.8294911351497897, 'MAPE_pct': np.float64(19.00432335818985), 'within_5s_pct': np.float64(12.345679012345679), 'within_10s_pct': np.float64(29.629629629629626), 'within_15s_pct': np.float64(43.20987654320987), 'within_30s_pct': np.float64(71.60493827160494), 'within_60s_pct': np.float64(93.20987654320987), 'bias_sec': np.float64(-7.6562943260082506), 'p90_absolute_sec': np.float64(54.460364275159385)}


## 5. Training-only search on the expanded dataset
Each complete observed trip group is held out in turn. October groups remain partially observed; their small sizes are shown in every result table. This is an offline group test and may use October training data when testing June.

Use three internal training-group folds. Test two common feature sets: EnhancedNoClock and MotionGeometry. Keep the same selected feature set for all four models in each outer fold. Select settings using mean internal-trip success within 30 seconds, then MAE. For Random Forest, compare ordinary and interval-based predictions inside the same training-only search. This improves on choosing forest settings for ordinary means and applying the interval rule afterwards.

Candidate lists are fixed before this run. Keep every outer test observation, including difficult cases. These data have now been used for method development, so the expanded results still need confirmation on more days.

In [13]:
SEARCH_VARIANTS={'Linear Regression':['Default'],'Random Forest':['Default','Deep','Smooth','Subsample'],
 'XGBoost':['Default','Shallow','Penalized'],'LightGBM':['Default','Flexible','Absolute']}
search_records=[];selected_records=[];expanded_predictions=[];expanded_folds=[]
for outer in trips:
    train=eta[eta.trip_id!=outer];test=eta[eta.trip_id==outer]
    assert not set(train.event_id)&set(test.event_id)
    candidates=[]
    inner_splits=list(GroupKFold(n_splits=3).split(train,groups=train.trip_id))
    for features in ['EnhancedNoClock','MotionGeometry']:
        for name in MODEL_NAMES:
            for variant in SEARCH_VARIANTS[name]:
                scores={'Ordinary':[],'Maximum-window':[]} if name=='Random Forest' else {'Ordinary':[]}
                for i,j in inner_splits:
                    tr=train.iloc[i];va=train.iloc[j]
                    assert not set(tr.event_id)&set(va.event_id)
                    p,fitted=fit_predict(tr,va,name,features,variant)
                    policies={'Ordinary':p}
                    if name=='Random Forest':policies['Maximum-window']=forest_tolerance_predict(fitted,tr,va,features)
                    for policy,pred in policies.items():
                        for trip in va.trip_id.unique():
                            mask=(va.trip_id==trip).to_numpy()
                            scores[policy].append(metrics(va.loc[mask,'target_eta_sec'],pred[mask]))
                for policy,m in scores.items():
                    candidates.append({'outer_test_trip':int(outer),'features':features,'model':name,'variant':variant,'policy':policy,
                        **pd.DataFrame(m).mean().to_dict()})
            print('Search outer group',outer,features,name,'complete.',flush=True)
    candidates=pd.DataFrame(candidates);search_records.append(candidates)
    winners=candidates.sort_values(['within_30s_pct','MAE_sec'],ascending=[False,True]).groupby(['features','model'],sort=False).head(1)
    shared=winners.groupby('features')[['within_30s_pct','MAE_sec']].mean().sort_values(['within_30s_pct','MAE_sec'],ascending=[False,True]).index[0]
    chosen=winners[winners.features==shared].set_index('model')
    for name in MODEL_NAMES:
        c=chosen.loc[name];pred,fitted=fit_predict(train,test,name,shared,c.variant)
        if c.policy=='Maximum-window':pred=forest_tolerance_predict(fitted,train,test,shared)
        pf=prediction_frame(test,pred,name,'Expanded nested',outer,shared,c.variant)
        pf['policy']=c.policy;pf['study_period']=test.study_period.to_numpy();pf['study_date']=test.study_date.to_numpy()
        expanded_predictions.append(pf)
        expanded_folds.append({'experiment':'Expanded nested','model':name,'test_trip':int(outer),'features':shared,'variant':c.variant,'policy':c.policy,
            'prediction_instances':len(test),'arrival_events':test.event_id.nunique(),**metrics(test.target_eta_sec,pred)})
        selected_records.append({'outer_test_trip':int(outer),'training_trip_groups':','.join(str(int(v)) for v in sorted(train.trip_id.unique())),
            'model':name,'features':shared,'variant':c.variant,'policy':c.policy,'inner_within_30s_pct':c.within_30s_pct,'inner_MAE_sec':c.MAE_sec})
    pd.concat(search_records,ignore_index=True).to_csv(OUT/'tables/training_only_search.csv',index=False)
    pd.DataFrame(selected_records).to_csv(OUT/'tables/training_only_choices.csv',index=False)
    pd.DataFrame(expanded_folds).to_csv(OUT/'tables/expanded_results_by_trip.csv',index=False)
    pd.concat(expanded_predictions,ignore_index=True).to_csv(OUT/'data/expanded_predictions.csv',index=False)
    print('OUTER GROUP',outer,'COMPLETE',flush=True)
predictions_list+=expanded_predictions;results+=expanded_folds

Search outer group 1 EnhancedNoClock Linear Regression complete.


Search outer group 1 EnhancedNoClock Random Forest complete.


Search outer group 1 EnhancedNoClock XGBoost complete.


Search outer group 1 EnhancedNoClock LightGBM complete.


Search outer group 1 MotionGeometry Linear Regression complete.


Search outer group 1 MotionGeometry Random Forest complete.


... (73 outputs left out)

Search outer group 108 MotionGeometry Random Forest complete.


Search outer group 108 MotionGeometry XGBoost complete.


Search outer group 108 MotionGeometry LightGBM complete.


OUTER GROUP 108 COMPLETE


## 6. Measure the effect of extra training data fairly
Keep the same seven June test trips and the same June test observations. Use fixed settings and features. Compare training on the other six June trips with training on those trips plus the October examples. This isolates the effect of adding October training data. It is an offline comparison, not a future-day forecast. Random Forest uses the same interval policy in both comparisons; the other models use their fixed June settings.

In [14]:
paired_rows=[]
for outer in sorted(JUNE.trip_id.unique()):
    test=JUNE[JUNE.trip_id==outer];jtrain=JUNE[JUNE.trip_id!=outer]
    for experiment,train in [('Same June test: June training',jtrain),('Same June test: extra October training',pd.concat([jtrain,OCTOBER],ignore_index=True))]:
        for name in MODEL_NAMES:
            variant=FROZEN_VARIANTS[name];pred,fitted=fit_predict(train,test,name,'EnhancedNoClock',variant)
            if name=='Random Forest':pred=forest_tolerance_predict(fitted,train,test,'EnhancedNoClock')
            record(test,pred,name,experiment,'EnhancedNoClock',variant)
    print('Paired June comparison group',outer,'complete.',flush=True)

Paired June comparison group 1 complete.


Paired June comparison group 2 complete.


Paired June comparison group 3 complete.


Paired June comparison group 4 complete.


Paired June comparison group 5 complete.


Paired June comparison group 6 complete.


Paired June comparison group 7 complete.


## 7. Results, figures and dashboard
Report later-day, expanded trip-based and same-June-test comparisons separately. The target is never changed to improve the headline result. Show all four models, test counts, arrival counts, MAE, RMSE and R². Several GPS observations can refer to one arrival.

In [15]:
predictions=pd.concat(predictions_list,ignore_index=True)
fold_results=pd.DataFrame(results)
summary_rows=[]
for (experiment,name),p in predictions.groupby(['experiment','model']):
    for scope,part in [('All eligible observations',p),('June',p[p.study_period=='June']),('October',p[p.study_period=='October'])]:
        if part.empty:continue
        trip_scores=[metrics(g.target_eta_sec,g.predicted_eta_sec) for _,g in part.groupby('trip_id')]
        summary_rows.append({'experiment':experiment,'model':name,'scope':scope,'prediction_instances':len(part),
            'arrival_events':part.event_id.nunique(),'test_groups':part.trip_id.nunique(),
            **metrics(part.target_eta_sec,part.predicted_eta_sec),
            'mean_trip_within_30s_pct':np.mean([r['within_30s_pct'] for r in trip_scores]),
            'min_trip_within_30s_pct':min(r['within_30s_pct'] for r in trip_scores),
            'sd_trip_MAE_sec':np.std([r['MAE_sec'] for r in trip_scores],ddof=1) if len(trip_scores)>1 else 0.})
all_summary=pd.DataFrame(summary_rows)
summary=all_summary[all_summary.scope=='All eligible observations'].copy()
summary['meets_80pct']=summary.within_30s_pct>=80
summary.to_csv(OUT/'tables/model_summary.csv',index=False)
all_summary.to_csv(OUT/'tables/model_summary_by_period.csv',index=False)
predictions.to_csv(OUT/'data/all_predictions.csv',index=False)
fold_results.to_csv(OUT/'tables/all_results_by_trip.csv',index=False)
best=summary[summary.experiment=='Expanded nested'].sort_values(['within_30s_pct','MAE_sec'],ascending=[False,True]).iloc[0]
best_model=best['model']
best_predictions=predictions[(predictions.experiment=='Expanded nested')&(predictions.model==best_model)].copy()
for key in ['checkpoint','trip_id']:
    rows=[]
    for value,g in best_predictions.groupby(key):rows.append({key:value,'observations':len(g),'arrival_events':g.event_id.nunique(),**metrics(g.target_eta_sec,g.predicted_eta_sec)})
    pd.DataFrame(rows).to_csv(OUT/'tables'/f'expanded_errors_by_{key}.csv',index=False)
best_predictions.sort_values('absolute_error_sec',ascending=False).head(40).to_csv(OUT/'tables/largest_expanded_errors.csv',index=False)
display(summary[['experiment','model','prediction_instances','arrival_events','within_30s_pct','MAE_sec','RMSE_sec','R2']].round(3))
print('BEST EXPANDED:',best_model,round(best.within_30s_pct,3),'percent within 30 seconds',flush=True)

,experiment,model,prediction_instances,arrival_events,within_30s_pct,MAE_sec,RMSE_sec,R2
0,Expanded nested,LightGBM,2066,88,70.281,25.426,36.677,0.780
3,Expanded nested,Linear Regression,2066,88,61.713,34.160,81.882,-0.097
6,Expanded nested,Random Forest,2066,88,77.541,25.488,36.945,0.777
9,Expanded nested,XGBoost,2066,88,71.636,24.928,35.877,0.789
12,Later-day fixed,LightGBM,162,11,71.605,24.311,32.391,0.829
14,Later-day fixed,Linear Regression,162,11,65.432,51.950,221.986,-7.008
16,Later-day fixed,Random Forest,162,11,70.988,23.911,31.180,0.842
18,Later-day fixed,XGBoost,162,11,71.605,24.052,31.632,0.837
20,Later-day tolerance forest,LightGBM,162,11,71.605,24.311,32.391,0.829
22,Later-day tolerance forest,Linear Regression,162,11,65.432,51.950,221.986,-7.008


BEST EXPANDED: Random Forest 77.541 percent within 30 seconds


## 8. Training-event sensitivity
Keep each test group unchanged while reducing whole training arrival events from 100% to 10%. Use three fixed random seeds below 100%. Use EnhancedNoClock and Default settings for all models at all levels. This experiment measures training-event availability, not reduced GPS recording frequency or energy savings. Give each observed test group equal weight in the summary, including the small October groups.

In [16]:
sensitivity_rows=[]
for outer in trips:
    train=eta[eta.trip_id!=outer];test=eta[eta.trip_id==outer]
    ids=sorted(train.event_id.unique())
    for pct in range(100,0,-10):
        for seed in ([11] if pct==100 else [11,23,37]):
            chosen=np.random.default_rng(seed).permutation(ids)[:max(1,int(np.ceil(len(ids)*pct/100)))]
            tr=train[train.event_id.isin(chosen)]
            assert not set(tr.event_id)&set(test.event_id)
            for name in MODEL_NAMES:
                pred,_=fit_predict(tr,test,name,'EnhancedNoClock','Default')
                sensitivity_rows.append({'test_trip':int(outer),'model':name,'training_pct':pct,'seed':seed,
                    'training_events':tr.event_id.nunique(),'test_instances':len(test),'test_events':test.event_id.nunique(),**metrics(test.target_eta_sec,pred)})
    pd.DataFrame(sensitivity_rows).to_csv(OUT/'tables/training_sensitivity_raw.csv',index=False)
    print('Sensitivity test group',outer,'complete.',flush=True)
raw=pd.DataFrame(sensitivity_rows)
per_trip=raw.groupby(['model','training_pct','test_trip'])[['MAE_sec','within_30s_pct','R2']].mean().reset_index()
sensitivity=per_trip.groupby(['model','training_pct']).agg(mean_MAE_sec=('MAE_sec','mean'),mean_within_30s_pct=('within_30s_pct','mean'),mean_R2=('R2','mean')).reset_index()
sensitivity.to_csv(OUT/'tables/training_sensitivity_summary.csv',index=False)

Sensitivity test group 1 complete.


Sensitivity test group 2 complete.


Sensitivity test group 3 complete.


Sensitivity test group 4 complete.


Sensitivity test group 5 complete.


Sensitivity test group 6 complete.


Sensitivity test group 7 complete.


Sensitivity test group 107 complete.


Sensitivity test group 108 complete.


## 9. Speed scenarios and simple baselines
The speed limit remains undocumented. Report illustrative thresholds only. Speed uses the previous km/h convention; this is not separately confirmed for the October export. The speed data shown cover the available records of the two labelled vehicles.

Compare machine learning with distance/current speed and training-mean ETA under the same trip splits. Report speed-baseline coverage when current speed is zero.

In [17]:
def speed_episodes(frame,threshold):
    rows=[]
    for unit,g in frame.groupby('vehicle_id'):
        g=g.sort_values('timestamp').copy()
        above=g.speed_kmh>threshold
        starts=above & ((~above.shift(fill_value=False))|(g.timestamp.diff().dt.total_seconds()>30))
        ids=starts.cumsum()
        for _,e in g[above].groupby(ids[above]):
            rows.append({'vehicle_id':unit,'threshold_kmh':threshold,'start_time':e.timestamp.iloc[0],
                'end_time':e.timestamp.iloc[-1],'observed_duration_sec':(e.timestamp.iloc[-1]-e.timestamp.iloc[0]).total_seconds(),
                'observations':len(e),'peak_speed_kmh':e.speed_kmh.max(),'mean_speed_kmh':e.speed_kmh.mean()})
    return pd.DataFrame(rows,columns=['vehicle_id','threshold_kmh','start_time','end_time','observed_duration_sec','observations','peak_speed_kmh','mean_speed_kmh'])
threshold_rows=[]
for limit in [20,30,40,50]:
    episodes=speed_episodes(gps,limit)
    threshold_rows.append({'scenario_threshold_kmh':limit,'exceeding_observations':int((gps.speed_kmh>limit).sum()),
        'exceeding_pct':100*(gps.speed_kmh>limit).mean(),'observed_episodes':len(episodes),'interpretation':'Illustrative threshold, not a verified legal limit'})
    episodes.to_csv(OUT/'tables'/f'speed_scenario_{limit}kmh_episodes.csv',index=False)
speed_scenarios=pd.DataFrame(threshold_rows)
speed_scenarios.to_csv(OUT/'tables'/'speed_threshold_scenarios.csv',index=False)
if SPEED_LIMIT_KMH is None:
    speed_status='Legal speeding result unavailable: applicable speed limit has not been supplied.'
else:
    assert SPEED_LIMIT_KMH>0
    speed_episodes(gps,SPEED_LIMIT_KMH).to_csv(OUT/'tables'/'speeding_episodes_configured_limit.csv',index=False)
    speed_status=f'Configured limit: {SPEED_LIMIT_KMH} km/h. Its applicability must be documented.'
print(speed_status);display(speed_scenarios)

Legal speeding result unavailable: applicable speed limit has not been supplied.


,scenario_threshold_kmh,exceeding_observations,exceeding_pct,observed_episodes,interpretation
0,20,321,3.822339,187,"Illustrative threshold, not a verified legal l..."
1,30,35,0.416766,20,"Illustrative threshold, not a verified legal l..."
2,40,1,0.011908,1,"Illustrative threshold, not a verified legal l..."
3,50,0,0.000000,0,"Illustrative threshold, not a verified legal l..."


In [18]:
baseline_rows=[]
for outer in trips:
    train=eta[eta.trip_id!=outer];test=eta[eta.trip_id==outer]
    for method in ['Training mean','Distance/current speed']:
        subset=test if method=='Training mean' else test[test.speed_kmh>0]
        pred=np.full(len(subset),train.target_eta_sec.mean()) if method=='Training mean' else subset.distance_centroid_km.to_numpy()/subset.speed_kmh.to_numpy()*3600
        baseline_rows.append({'test_group':int(outer),'method':method,'test_instances':len(subset),'coverage_pct':100*len(subset)/len(test),**metrics(subset.target_eta_sec,pred)})
pd.DataFrame(baseline_rows).to_csv(OUT/'tables/baseline_results.csv',index=False)

## 10. English figures and offline dashboard
The dashboard contains its data and plotting library. Open the HTML in a browser after downloading it; a file-storage preview may not run JavaScript. Keep the embedded plotting library unchanged when editing report wording.

In [19]:
def render_figures():
    matched=pd.read_csv(OUT/'data/matched_arrival_events.csv')
    nested=summary[summary.experiment=='Expanded nested'].sort_values(['within_30s_pct','MAE_sec'],ascending=[False,True]).iloc[0]
    best_model=nested['model']
    best_predictions=predictions[(predictions.experiment=='Expanded nested')&(predictions.model==best_model)]
    COLORS={'Linear Regression':'#8b97a5','Random Forest':'#187e82','XGBoost':'#ef9f36','LightGBM':'#536bb7'}
    def save_figure(fig,name):
        fig.tight_layout()
        for suffix in ['png','svg']:fig.savefig(OUT/'figures'/f'{name}.{suffix}',bbox_inches='tight')
        plt.close(fig)
    for experiment,filename,title in [('Later-day tolerance forest','01_later_day','June training → October test'),('Expanded nested','02_expanded_comparison','Expanded study — held-out observed trip groups')]:
        s=summary[summary.experiment==experiment].set_index('model').loc[MODEL_NAMES]
        fig,ax=plt.subplots(figsize=(9,4));ax.bar(MODEL_NAMES,s.within_30s_pct,color=[COLORS[n] for n in MODEL_NAMES])
        ax.axhline(80,color='#b04e43',ls='--',label='Target: 80%');ax.set_ylim(0,100);ax.set_ylabel('Predictions within ±30 s (%)');ax.set_title(title);ax.legend()
        save_figure(fig,filename)
    fig,ax=plt.subplots(figsize=(10,4));x=np.arange(4);width=.35
    for j,exp in enumerate(['Same June test: June training','Same June test: extra October training']):
        s=summary[summary.experiment==exp].set_index('model').loc[MODEL_NAMES]
        ax.bar(x+(j-.5)*width,s.within_30s_pct,width,label=exp)
    ax.axhline(80,color='#b04e43',ls='--');ax.set_xticks(x,MODEL_NAMES);ax.set_ylim(0,100);ax.set_ylabel('Predictions within ±30 s (%)');ax.set_title('Effect of adding October data — same 1,904 June test observations');ax.legend(fontsize=8)
    save_figure(fig,'03_same_test_extra_training')
    fig,axes=plt.subplots(1,3,figsize=(14,4))
    for ax,col,label in zip(axes,['MAE_sec','RMSE_sec','R2'],['MAE (s)','RMSE (s)','R²']):
        s=summary[summary.experiment=='Expanded nested'].set_index('model').loc[MODEL_NAMES]
        ax.bar(MODEL_NAMES,s[col],color=[COLORS[n] for n in MODEL_NAMES]);ax.set_ylabel(label);ax.tick_params(axis='x',rotation=25)
    fig.suptitle('Expanded nested comparison');save_figure(fig,'04_model_error_metrics')
    fig,axes=plt.subplots(1,2,figsize=(12,4))
    axes[0].scatter(best_predictions.target_eta_sec,best_predictions.predicted_eta_sec,s=6,alpha=.35,color=COLORS[best_model]);axes[0].plot([0,350],[0,350],ls='--',color='gray');axes[0].set_xlabel('Actual ETA (s)');axes[0].set_ylabel('Predicted ETA (s)')
    axes[1].hist(best_predictions.arrival_error_sec,bins=40,color=COLORS[best_model]);axes[1].set_xlabel('Predicted − actual arrival (s)');axes[1].set_ylabel('Observations')
    fig.suptitle(best_model+' — expanded held-out results');save_figure(fig,'05_predictions_and_errors')
    fig,ax=plt.subplots(figsize=(10,4))
    for name in MODEL_NAMES:
        s=fold_results[(fold_results.experiment=='Expanded nested')&(fold_results.model==name)].sort_values('test_trip')
        ax.plot(s.test_trip.astype(str),s.within_30s_pct,marker='o',label=name,color=COLORS[name])
    ax.axhline(80,color='#b04e43',ls='--');ax.set_ylim(0,100);ax.set_xlabel('Test group (107/108: partial October trips 7/8)');ax.set_ylabel('Within ±30 s (%)');ax.legend(fontsize=8);save_figure(fig,'06_trip_variation')
    fig,axes=plt.subplots(1,2,figsize=(12,4))
    for name in MODEL_NAMES:
        s=sensitivity[sensitivity.model==name].sort_values('training_pct')
        axes[0].plot(s.training_pct,s.mean_MAE_sec,marker='o',label=name,color=COLORS[name]);axes[1].plot(s.training_pct,s.mean_within_30s_pct,marker='o',label=name,color=COLORS[name])
    axes[0].set_yscale('log');axes[0].set_ylabel('Mean group MAE (s; logarithmic scale)');axes[1].set_ylabel('Mean group within ±30 s (%)')
    for ax in axes:ax.invert_xaxis();ax.set_xlabel('Training events retained (%)')
    axes[0].legend(fontsize=8);fig.suptitle('Training-event sensitivity — unchanged tests, three seeds');save_figure(fig,'07_training_sensitivity')
    fig,axes=plt.subplots(1,2,figsize=(12,4))
    axes[0].hist(matched.signed_diff_sec,bins=25,color='#187e82');axes[0].set_xlabel('GPS zone entry − logbook arrival (s)');axes[0].set_ylabel('Matched arrivals')
    axes[1].bar(speed_scenarios.scenario_threshold_kmh.astype(str),speed_scenarios.exceeding_observations,color='#ef9f36');axes[1].set_xlabel('Illustrative speed threshold (km/h)');axes[1].set_ylabel('Exceeding observations')
    fig.suptitle('Matching and speed scenarios — legal speed limit unavailable');save_figure(fig,'08_matching_and_speed')

    if 'GPS history fixed' in set(summary.experiment):
        fig,ax=plt.subplots(figsize=(10,4));x=np.arange(4);width=.35
        for j,experiment in enumerate(['Expanded nested','GPS history fixed']):
            ss=summary[summary.experiment==experiment].set_index('model').loc[MODEL_NAMES]
            ax.bar(x+(j-.5)*width,ss.within_30s_pct,width,label=experiment)
        ax.axhline(80,color='#b04e43',ls='--',label='80% target');ax.set_xticks(x,MODEL_NAMES);ax.set_ylim(0,100)
        ax.set_ylabel('Predictions within ±30 s (%)');ax.set_title('GPS history — same 2,066 observations');ax.legend(fontsize=8)
        save_figure(fig,'09_GPS_history_comparison')

render_figures()

In [20]:
def render_dashboard():
    import html
    dashboard_best=summary[summary.prediction_instances==summary.prediction_instances.max()].sort_values(['within_30s_pct','MAE_sec','selection_priority'],ascending=[False,True,True]).iloc[0]
    dashboard_cases={dashboard_best['experiment'],'Expanded nested','GPS history fixed','Later-day tolerance forest','Same June test: June training','Same June test: extra October training'}
    dashboard_predictions=predictions[predictions.experiment.isin(dashboard_cases)]
    payload={
     'summary':json.loads(summary.to_json(orient='records')),
     'folds':json.loads(fold_results.to_json(orient='records')),
     'predictions':json.loads(dashboard_predictions[['experiment','model','trip_id','checkpoint','event_id','timestamp','actual_arrival','target_eta_sec','predicted_eta_sec','absolute_error_sec','arrival_error_sec']].to_json(orient='records',date_format='iso',double_precision=15)),
     'sensitivity':json.loads(sensitivity.to_json(orient='records')),
     'quality':json.loads(quality.to_json(orient='records')),
     'speed':json.loads(gps[['timestamp','speed_kmh']].to_json(orient='records',date_format='iso')),
     'speed_scenarios':json.loads(speed_scenarios.to_json(orient='records')),
     'route':{'longitude':gps.longitude.round(6).tolist(),'latitude':gps.latitude.round(6).tolist()},
     'zones':[{'name':k,'longitude':list(p.exterior.xy[0]),'latitude':list(p.exterior.xy[1])} for k,p in zone_polys.items()],
     'best_model':dashboard_best['model'],'default_experiment':dashboard_best['experiment'],'speed_status':speed_status}
    dashboard_template = r"""<!doctype html>
    <html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
    <title>Hanine | Bus arrival research</title><script>__PLOTLY__</script>
    <style>
    *{box-sizing:border-box}html{scroll-behavior:smooth;scroll-padding-top:100px}body{margin:0;background:#f3f6f7;color:#193342;font:15px/1.55 'Segoe UI',Arial,sans-serif}header{position:sticky;top:0;z-index:10;background:#153848;color:white;padding:14px 5%;display:flex;gap:30px;justify-content:space-between;align-items:center}header strong{font-size:18px}nav{display:flex;gap:18px;flex-wrap:wrap}nav a{color:#dce9ed;text-decoration:none;font-size:13px}main{max-width:1280px;margin:auto;padding:30px 28px 60px}h1{font-size:36px;margin:0;letter-spacing:-1px}h2{font-size:22px;margin:0 0 16px}h3{margin:0 0 5px;font-size:16px}.intro{max-width:850px;color:#526975}.eyebrow{letter-spacing:2px;font-size:11px;color:#187e82;text-transform:uppercase;font-weight:700}.filters{display:flex;gap:16px;flex-wrap:wrap;background:white;padding:18px;border-radius:10px;border:1px solid #dce5e9;margin:24px 0}label{font-size:12px;color:#526975;display:flex;flex-direction:column;gap:6px}select{padding:9px 30px 9px 10px;border:1px solid #c8d6dd;border-radius:5px;background:white;color:#193342;min-width:160px}.kpis{display:grid;grid-template-columns:repeat(4,1fr);gap:16px}.card{background:white;border:1px solid #dce5e9;border-radius:10px;padding:20px}.kpi-value{font-size:31px;font-weight:650;margin:8px 0 2px;letter-spacing:-1px}.muted{font-size:12px;color:#607582}.status{margin:16px 0 24px;padding:14px 18px;background:#fff1df;border-left:4px solid #d78c29;border-radius:3px}.grid{display:grid;grid-template-columns:1fr 1fr;gap:18px}.plot{height:355px}.wide{grid-column:1/-1}section{margin-top:34px}table{width:100%;border-collapse:collapse;font-size:13px}th{text-align:left;background:#eef3f5;padding:10px;color:#526975}td{padding:9px 10px;border-bottom:1px solid #edf1f3}.table-wrap{overflow:auto}.pill{font-size:12px;color:#187e82;font-weight:600}.note{font-size:13px;color:#526975;margin-top:12px}.download{border:0;border-radius:5px;background:#187e82;color:white;padding:10px 16px;cursor:pointer}.foot{color:#607582;font-size:12px;margin-top:32px}@media(max-width:800px){header{align-items:flex-start;gap:10px;flex-direction:column}h1{font-size:28px}.kpis{grid-template-columns:1fr 1fr}.grid{grid-template-columns:1fr}.wide{grid-column:auto}main{padding:22px 14px}.plot{height:330px}}@media(max-width:420px){.kpis{grid-template-columns:1fr}.filters label,select{width:100%}}
    </style></head><body>
    <header><strong>HANINE / Transport research</strong><nav><a href="#overview">Overview</a><a href="#validation">Trip validation</a><a href="#errors">Prediction errors</a><a href="#sensitivity">Sensitivity</a><a href="#speed">Speed</a><a href="#data">Data & method</a></nav></header>
    <main><div class="eyebrow">GPS-based ETA · June and October study</div><h1>How close is the predicted arrival?</h1>
    <p class="intro">Compare ETA methods on trips excluded from training. The main objective is <strong>80% of predictions within ±30 seconds</strong> of the recorded arrival. Explore the full result, including weak trips and errors.</p>
    <div class="filters"><label>Experiment<select id="experiment"></select></label><label>Model<select id="model"></select></label><label>Held-out trip<select id="trip"></select></label><label>Control point<select id="checkpoint"></select></label></div>
    <section id="overview"><div class="kpis"><div class="card"><h3>Within ±30 seconds</h3><div id="success" class="kpi-value"></div><div class="muted">Fixed target: at least 80%</div></div><div class="card"><h3>Mean absolute error</h3><div id="mae" class="kpi-value"></div><div class="muted">Average error magnitude</div></div><div class="card"><h3>RMSE</h3><div id="rmse" class="kpi-value"></div><div class="muted">More weight on large errors</div></div><div class="card"><h3>R²</h3><div id="r2" class="kpi-value"></div><div class="muted">Regression score, not accuracy</div></div></div><div id="status" class="status"></div>
    <div class="card"><h2>Model comparison</h2><div id="modelChart" class="plot"></div><div class="table-wrap" id="modelTable"></div><p class="note">Comparison uses the selected experiment, trip and control point. Counts are GPS prediction instances; several instances may belong to the same arrival event.</p></div></section>
    <section id="validation"><h2>Does performance hold across trips?</h2><div class="grid"><div class="card"><div id="tripChart" class="plot"></div></div><div class="card"><div id="toleranceChart" class="plot"></div></div></div><p class="note">Trip chart shows the observed test groups for the selected experiment and model; 107/108 are partial October trips 7/8, with the checkpoint filter applied. Tolerance exploration does not change the ±30-second primary criterion.</p></section>
    <section id="errors"><h2>Arrival predictions and their errors</h2><div class="grid"><div class="card"><div id="scatter" class="plot"></div></div><div class="card"><div id="hist" class="plot"></div></div><div class="card wide"><div id="checkpointChart" class="plot"></div><h3>Largest errors in the current selection</h3><div class="table-wrap" id="errorTable"></div><p><button id="download" class="download">Download filtered predictions (CSV)</button></p><details id="csvPreview" hidden><summary id="csvSummary">CSV export preview</summary><p class="note">If your browser blocks the download, copy the text below into a UTF-8 .csv file. The preview contains every filtered prediction.</p><textarea id="csvText" readonly aria-label="Filtered predictions CSV" style="width:100%;height:160px;font:12px monospace"></textarea></details></div></div></section>
    <section id="sensitivity"><h2>What happens with less training data?</h2><p class="intro">Whole training arrival events are sampled from 100% down to 10%. Each observed test group stays fixed. Three seeds are averaged, then the nine groups receive equal weight. These charts use fixed Default models with EnhancedNoClock features.</p><div class="grid"><div class="card"><div id="sensMAE" class="plot"></div></div><div class="card"><div id="sensSuccess" class="plot"></div></div></div></section>
    <section id="speed"><h2>Speed and threshold exceedances</h2><div id="speedStatus" class="status"></div><div class="grid"><div class="card"><div id="speedChart" class="plot"></div></div><div class="card"><div id="speedScenario" class="plot"></div></div></div><p class="note">20/30/40/50 km/h are illustrative thresholds. They do not establish legal violations. Speed uses the previous km/h convention; the October export unit is not separately confirmed.</p></section>
    <section id="data"><h2>Data, route and interpretation</h2><div class="grid"><div class="card"><div id="routeChart" class="plot"></div></div><div class="card"><h3>Source audit</h3><div id="qualityTable"></div><p class="note">GPS vehicle r4-u7 is matched to logbook unit 1. This reported correspondence has not been checked with the data provider.</p></div><div class="card wide"><h3>How to interpret this study</h3><p>GPS records → cleaning → zone entry detection → one-to-one logbook matching → trailing motion features → ETA labels → trip-based model comparison → sensitivity and error analysis.</p><p>Selected comparisons are shown here; all tested experiments remain in model_summary.csv and all_predictions.csv. All predictions shown are held out by observed trip group. The later-day test trains only on June with settings fixed beforehand. The expanded comparison selects shared features and model settings using training groups only. Same-June-test comparisons isolate the effect of adding October training examples. October groups cover only part of their trips.</p><p>Two dates and eleven new matched arrivals are insufficient to establish fleet-wide performance. October GPS U16 is provisionally matched to logbook unit 16. October speed uses the previous km/h convention. The new arrival-recording system and clock accuracy are undocumented. Trip grouping and the 30–300-second study horizon are retrospective. The supplied destination is known to the predictor. Speed limits, passenger satisfaction data and validated GPS-energy costs are absent.</p></div></div></section>
    <p class="foot">Generated by the accompanying reproducible notebook. All figures and tables use English labels. The dashboard contains its data and plotting library and works offline.</p></main>
    <script id="research-data" type="application/json">__DATA__</script><script>
    const D=JSON.parse(document.getElementById('research-data').textContent), $=id=>document.getElementById(id);
    const names=['Linear Regression','Random Forest','XGBoost','LightGBM'];const colors=['#8b97a5','#187e82','#ef9f36','#536bb7'];
    const mean=a=>a.reduce((s,x)=>s+x,0)/a.length;const fmt=(x,n=2)=>Number.isFinite(x)?x.toFixed(n):'Unavailable';
    function measure(p){if(!p.length)return {n:0,events:0,success:NaN,mae:NaN,rmse:NaN,r2:NaN};let y=p.map(r=>r.target_eta_sec),e=p.map(r=>r.predicted_eta_sec-r.target_eta_sec),ss=y.reduce((s,x)=>s+(x-mean(y))**2,0);return {n:p.length,events:new Set(p.map(r=>r.event_id)).size,success:100*e.filter(x=>Math.abs(x)<=30).length/e.length,mae:mean(e.map(Math.abs)),rmse:Math.sqrt(mean(e.map(x=>x*x))),r2:ss?1-e.reduce((s,x)=>s+x*x,0)/ss:NaN}}
    const layout=(title,x,y)=>({title:{text:title,font:{size:16}},font:{family:'Segoe UI, Arial',color:'#193342'},paper_bgcolor:'white',plot_bgcolor:'white',margin:{t:55,l:58,r:20,b:65},xaxis:{title:{text:x},gridcolor:'#edf1f3'},yaxis:{title:{text:y},gridcolor:'#edf1f3'},legend:{orientation:'h',y:-.25}});
    const plot=(id,traces,l)=>Plotly.react(id,traces,l,{responsive:true,displaylogo:false});
    function options(id,arr,selected){$(id).replaceChildren(...arr.map(x=>{let o=document.createElement('option');o.value=x;o.textContent=x;return o}));if(selected)$(id).value=selected}
    options('experiment',Array.from(new Set(D.predictions.map(p=>p.experiment))));$('experiment').value=D.default_experiment;options('trip',['All trips',...Array.from(new Set(D.predictions.map(p=>p.trip_id))).sort((a,b)=>a-b).map(String)]);options('checkpoint',['All control points',...Array.from(new Set(D.predictions.map(p=>p.checkpoint))).sort()]);
    function modelOptions(){let ns=Array.from(new Set(D.predictions.filter(p=>p.experiment===$('experiment').value).map(p=>p.model)));options('model',ns,ns.includes(D.best_model)?D.best_model:ns[0])}modelOptions();
    function filter(p,withModel=true,withTrip=true){return p.experiment===$('experiment').value&&(!withModel||p.model===$('model').value)&&(!withTrip||$('trip').value==='All trips'||String(p.trip_id)===$('trip').value)&&($('checkpoint').value==='All control points'||p.checkpoint===$('checkpoint').value)}
    function table(id,headers,rows){let t=document.createElement('table'),head=t.createTHead(),hr=head.insertRow();headers.forEach(h=>{let c=document.createElement('th');c.textContent=h;hr.append(c)});let body=t.createTBody();rows.forEach(row=>{let tr=body.insertRow();row.forEach(v=>tr.insertCell().textContent=v)});$(id).replaceChildren(t)}
    let current=[];
    function refresh(){current=D.predictions.filter(p=>filter(p));let m=measure(current);$('success').textContent=fmt(m.success)+'%';$('mae').textContent=fmt(m.mae)+' s';$('rmse').textContent=fmt(m.rmse)+' s';$('r2').textContent=fmt(m.r2,3);$('status').textContent=(m.n?(m.success>=80?'The selected predictions meet':'The selected predictions do not meet')+' the 80% target within ±30 seconds. '+m.n+' prediction instances from '+m.events+' arrival events.':'No predictions match these filters.');
    let ns=Array.from(new Set(D.predictions.filter(p=>filter(p,false)).map(p=>p.model))),ms=ns.map(n=>measure(D.predictions.filter(p=>filter(p,false)&&p.model===n)));
    plot('modelChart',[{type:'bar',x:ns,y:ms.map(m=>m.success),marker:{color:ns.map(n=>colors[names.indexOf(n)]||'#7d929e')}}],{...layout('Predictions within ±30 seconds','Model','Success rate (%)'),yaxis:{title:{text:'Success rate (%)'},range:[0,100]},shapes:[{type:'line',xref:'paper',x0:0,x1:1,y0:80,y1:80,line:{color:'#b04e43',dash:'dash'}}]});
    table('modelTable',['Model','MAE (s)','RMSE (s)','R²','Within ±30 s','Instances','Events'],ns.map((n,i)=>[n,fmt(ms[i].mae),fmt(ms[i].rmse),fmt(ms[i].r2,3),fmt(ms[i].success)+'%',ms[i].n,ms[i].events]));
    let ts=Array.from(new Set(D.predictions.filter(p=>filter(p,true,false)).map(p=>p.trip_id))).sort((a,b)=>a-b),tm=ts.map(t=>measure(D.predictions.filter(p=>filter(p,true,false)&&p.trip_id===t)));
    plot('tripChart',[{type:'scatter',mode:'lines+markers',x:ts,y:tm.map(m=>m.success),line:{color:'#187e82'}}],{...layout('Success rate by held-out trip','Test trip','Within ±30 seconds (%)'),yaxis:{title:{text:'Within ±30 seconds (%)'},range:[0,100]}});
    let to=[5,10,15,30,60,90,120];plot('toleranceChart',[{type:'scatter',mode:'lines+markers',x:to,y:to.map(t=>current.length?100*current.filter(p=>p.absolute_error_sec<=t).length/current.length:null),line:{color:'#187e82'}}],layout('Tolerance exploration','Absolute error tolerance (s)','Predictions within tolerance (%)'));
    plot('scatter',[{type:'scattergl',mode:'markers',x:current.map(p=>p.target_eta_sec),y:current.map(p=>p.predicted_eta_sec),marker:{size:4,opacity:.4,color:'#187e82'}},{type:'scatter',mode:'lines',x:[0,350],y:[0,350],line:{color:'#b04e43',dash:'dash'},name:'Perfect prediction'}],layout('Predicted versus actual ETA','Actual ETA (s)','Predicted ETA (s)'));
    plot('hist',[{type:'histogram',x:current.map(p=>p.arrival_error_sec),marker:{color:'#187e82'},nbinsx:40}],layout('Signed prediction errors','Predicted arrival − actual arrival (s)','Instances'));
    let cp=Array.from(new Set(current.map(p=>p.checkpoint))).sort();plot('checkpointChart',[{type:'bar',x:cp,y:cp.map(c=>measure(current.filter(p=>p.checkpoint===c)).mae),marker:{color:'#187e82'}}],layout('Errors by control point','Control point','MAE (s)'));
    table('errorTable',['Trip','Control point','Prediction time','Actual arrival','Predicted ETA (s)','Actual ETA (s)','Error (s)'],[...current].sort((a,b)=>b.absolute_error_sec-a.absolute_error_sec).slice(0,8).map(p=>[p.trip_id,p.checkpoint,p.timestamp.slice(11,19),p.actual_arrival.slice(11,19),fmt(p.predicted_eta_sec),fmt(p.target_eta_sec),fmt(p.arrival_error_sec)]));
    }
    for(let id of ['model','trip','checkpoint'])$(id).addEventListener('change',refresh);$('experiment').addEventListener('change',()=>{modelOptions();refresh()});
    for(let [id,col,title,unit] of [['sensMAE','mean_MAE_sec','Error with less training data','Mean trip MAE (s)'],['sensSuccess','mean_within_30s_pct','Success with less training data','Mean trip within ±30 s (%)']]){let traces=names.map((n,i)=>{let s=D.sensitivity.filter(p=>p.model===n).sort((a,b)=>a.training_pct-b.training_pct);return {type:'scatter',mode:'lines+markers',name:n,x:s.map(p=>p.training_pct),y:s.map(p=>p[col]),line:{color:colors[i]}}});plot(id,traces,{...layout(title,'Training arrival events retained (%)',unit),yaxis:{title:{text:unit+(id==='sensMAE'?' — logarithmic scale':'')},...(id==='sensMAE'?{type:'log'}:{})},xaxis:{title:{text:'Training arrival events retained (%)'},autorange:'reversed'}})}
    $('speedStatus').textContent=D.speed_status;plot('speedChart',[{type:'scatter',mode:'lines',x:D.speed.map(p=>p.timestamp),y:D.speed.map(p=>p.speed_kmh),line:{color:'#187e82',width:1}}],layout('Available speed observations','Source local time','Speed (km/h)'));
    plot('speedScenario',[{type:'bar',x:D.speed_scenarios.map(p=>String(p.scenario_threshold_kmh)),y:D.speed_scenarios.map(p=>p.exceeding_observations),marker:{color:'#ef9f36'}}],layout('Illustrative threshold exceedances','Scenario threshold (km/h)','Exceeding observations'));
    let route=[{type:'scatter',mode:'lines',x:D.route.longitude,y:D.route.latitude,line:{color:'#adb8c0',width:1},name:'GPS trajectory'},...D.zones.map(z=>({type:'scatter',mode:'lines',x:z.longitude,y:z.latitude,name:z.name,line:{width:1.5}}))];plot('routeChart',route,{...layout('Route and control zones','Longitude (degrees)','Latitude (degrees)'),showlegend:false});
    table('qualityTable',['Measure','Value'],D.quality.map(q=>[q.measure,q.value]));
    $('download').addEventListener('click',()=>{let keys=['experiment','model','trip_id','checkpoint','event_id','timestamp','actual_arrival','target_eta_sec','predicted_eta_sec','arrival_error_sec'];let quote=v=>'"'+String(v).replaceAll('"','""')+'"';let text=keys.join(',')+'\n'+current.map(p=>keys.map(k=>quote(p[k])).join(',')).join('\n');$('csvText').value=text;$('csvPreview').hidden=false;$('csvPreview').open=true;$('csvSummary').textContent='CSV ready: '+current.length+' predictions';let url=URL.createObjectURL(new Blob([text],{type:'text/csv'})),a=document.createElement('a');a.href=url;a.download='filtered_predictions.csv';document.body.append(a);a.click();a.remove();setTimeout(()=>URL.revokeObjectURL(url),30000)});
    refresh();window.researchDashboard={data:D,measure,refresh,getCurrent:()=>current};
    </script></body></html>"""
    dashboard=dashboard_template.replace('__PLOTLY__',get_plotlyjs()).replace('__DATA__',json.dumps(payload,ensure_ascii=False).replace('</','<\\/'))
    (OUT/'dashboard'/'Hanine_ETA_Dashboard.html').write_text(dashboard,encoding='utf-8')
    print('Offline dashboard:',OUT/'dashboard'/'Hanine_ETA_Dashboard.html')
    display(HTML('<p><strong>Dashboard generated.</strong> Open the HTML file in a browser. Its data and plotting library are embedded.</p>'))

render_dashboard()

Offline dashboard: C:\Users\LENOVO\Desktop\Hanine\work_update_20261028\outputs\dashboard\Hanine_ETA_Dashboard.html


## 11. Reading the results
Use the expanded trip-based comparison as the updated offline study, and the June-to-October test as separate evidence about a later date. Use the same-June-test tables to judge whether extra training examples improved the same old test population. Do not combine these scores into one claim. The speed unit and vehicle correspondence for the new export remain assumptions.

The previous 76.52% result used the June study and a different training-only selection procedure. Keep its version and evidence separate. The new search and tests have been specified in this notebook; they do not guarantee 80%.

In [21]:
report=[]
for exp in ['Later-day tolerance forest','Expanded nested','Same June test: June training','Same June test: extra October training']:
    report+=['## '+exp,'','| Modèle | Observations | Arrivées | Dans ±30 s | MAE (s) | RMSE (s) | R² |','|---|---:|---:|---:|---:|---:|---:|']
    for name in MODEL_NAMES:
        r=summary[(summary.experiment==exp)&(summary.model==name)].iloc[0]
        report.append(f'| {name} | {int(r.prediction_instances)} | {int(r.arrival_events)} | {r.within_30s_pct:.2f}% | {r.MAE_sec:.2f} | {r.RMSE_sec:.2f} | {r.R2:.3f} |')
    report.append('')
outcome='atteint' if best.within_30s_pct>=80 else 'non atteint'
guide=f"""# Mise à jour de l’étude ETA

Les deux nouveaux exports datent du 27 octobre 2026. Le GPS contient 14 500 lignes pour 29 véhicules, avec exactement 500 lignes par véhicule. Le journal contient 106 arrivées pour l’unité 16. Le GPS U16 couvre seulement 16:57:45–19:35:54. Onze arrivées disposent d’un passage GPS apparié. Les autres véhicules n’ont pas de journal d’arrivée fourni et ne deviennent pas des exemples d’entraînement.

La correspondance U16/unité 16 est soutenue par les passages aux points de contrôle, mais reste non confirmée par le fournisseur. La vitesse suit la convention précédente en km/h; l’unité du nouvel export n’est pas confirmée séparément. Aucun décalage d’horloge n’est appliqué. Les coordonnées du nouveau CSV sont déjà correctement nommées, contrairement aux anciens en-têtes Excel inversés. La documentation du système d’enregistrement des nouvelles arrivées est inconnue. Des horaires identiques soutiennent l’appariement, sans prouver l’indépendance des systèmes.

Le jeu actualisé contient {len(eta)} observations, {eta.event_id.nunique()} arrivées et {len(trips)} groupes de test. Les deux groupes d’octobre ne sont pas des tournées complètes. Leurs identifiants 107/108 représentent les parties observées des tournées 7/8. Les limites de tournée et l’horizon réel 30–300 secondes restent des règles d’étude rétrospective, pas des informations disponibles en direct.

Le meilleur résultat de la comparaison actualisée est {best_model} : {best.within_30s_pct:.2f}% dans trente secondes, MAE {best.MAE_sec:.2f} s. L’objectif 80% est **{outcome} pour cette comparaison**. Présenter séparément le test de la nouvelle date et les comparaisons sur les mêmes observations de juin.

"""+'\n'.join(report)+"""
## Ce qui change dans le mémoire

Décrire séparément les deux dates, véhicules et couvertures. Ajouter le tableau de couverture, puis le test juin → octobre, la validation par groupes et la comparaison sur les mêmes tests de juin. Mettre à jour les résultats, figures, discussion et conclusion ensemble. Ne pas appeler les 14 500 nouvelles lignes GPS des exemples ETA utilisables : seules les observations associées à une arrivée appariée ont une cible.

Le fichier model_summary.csv donne les résultats globaux par expérience; model_summary_by_period.csv distingue juin et octobre. all_results_by_trip.csv donne la variation par groupe. Les figures 01–03 distinguent les trois questions. Le dashboard permet les mêmes filtres. Le score historique 76,52% reste la référence de l’ancienne version, pas une preuve d’amélioration à cohorte identique.

## Limites

Une seule nouvelle date et onze arrivées ne suffisent pas à garantir 80% sur la flotte. La recherche a été affinée dans cette étude; davantage de journées indépendantes sont nécessaires. L’historique reste continu par véhicule et utilise uniquement le présent/passé. Les paramètres sont choisis dans les groupes d’entraînement, sans les labels du test externe. La sensibilité utilise trois graines sous 100%, avec les mêmes tests. Elle mesure moins d’événements d’entraînement, pas une économie d’énergie.

La limite de vitesse légale est inconnue; les scénarios de vitesse ne sont pas des infractions établies. Le notebook est exécuté localement; son exécution dans le cloud Colab reste non vérifiée. Les données brutes et l’ancienne version restent conservées.

## Fichiers à utiliser

- Hanine_ETA_Bus_v2.ipynb : seul notebook de programmation actualisé.
- input/ : cinq fichiers originaux nécessaires pour relancer l’étude.
- outputs/dashboard/Hanine_ETA_Dashboard.html : présentation interactive autonome, à télécharger puis ouvrir dans un navigateur.
- outputs/tables/model_summary.csv et all_results_by_trip.csv : résultats à citer.
- outputs/figures/ : PNG pour le mémoire et SVG des mêmes figures.
- outputs/data/ : données préparées, copies CSV aux en-têtes anglais et prédictions de test.

Les fichiers techniques peuvent rester en annexe numérique. Le mémoire complet n’est pas rédigé ici.
"""
(OUT/'documentation/Guide_mise_a_jour_FR.md').write_text(guide,encoding='utf-8')
provenance={'new_vehicle_mapping':'U16 to unit 16; suffix agreement and 11 matched passages; provider confirmation unavailable',
 'new_speed_unit':'Previous km/h convention assumed; not independently confirmed for new CSV',
 'new_arrival_recording_system':'Undocumented','new_clock_shift_applied_seconds':0,
 'new_labelled_arrivals':11,'new_GPS_vehicles':29,'new_GPS_rows_per_vehicle':500,
 'new_trip_coverage':'Partial trips 7 and 8','primary_tolerance_seconds':30,'primary_target_percent':80}
(OUT/'documentation/data_provenance.json').write_text(json.dumps(provenance,indent=2))
assert all(sha256(DATA_DIR/name)==h for name,h in source_hashes.items())
manifest={'source_hashes':source_hashes,'rows':len(eta),'arrival_events':int(eta.event_id.nunique()),'test_groups':len(trips),
 'expanded_best_model':best_model,'expanded_best_within_30s_pct':float(best.within_30s_pct),
 'outputs':{str(p.relative_to(OUT)):sha256(p) for p in OUT.rglob('*') if p.is_file() and p.name!='manifest.json'}}
(OUT/'documentation/manifest.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2))
print('Updated results and guide exported. Expanded 80% target:',outcome,flush=True)

Updated results and guide exported. Expanded 85% target: non atteint


## 12. Fixed-setting reference
The 100% training-data sensitivity runs also provide a nine-group reference with unchanged Default settings. Summarise those existing calculations without training the same models twice. Report mean, standard deviation, minimum and maximum MAE by observed test group.

In [1]:
from pathlib import Path
import os, sys, subprocess, importlib.util
try:
    IN_COLAB = importlib.util.find_spec('google.colab') is not None
except ModuleNotFoundError:
    IN_COLAB = False
if IN_COLAB:
    loaded_versions = {name:getattr(sys.modules.get(name),'__version__',None) for name in ['numpy','pandas']}
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
        'pandas==3.0.6', 'numpy==2.5.3', 'scipy==1.18.1', 'scikit-learn==1.9.1',
        'shapely==2.2.0', 'xgboost==3.4.1', 'lightgbm==4.7.0',
        'plotly==7.1.0', 'matplotlib==3.11.2', 'openpyxl==3.1.5'])
    expected_core={'numpy':'2.5.3','pandas':'3.0.6'}
    if any(v is not None and v!=expected_core[name] for name,v in loaded_versions.items()):
        raise RuntimeError('Dependencies installed. In Colab choose Runtime > Restart session, then Run all again. This prevents mixing old loaded libraries with newly installed versions.')
import json, re, hashlib, unicodedata, platform, warnings, time, copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from shapely.geometry import Point, Polygon, LineString, mapping
from shapely.ops import transform
from shapely import covers, points
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from threadpoolctl import threadpool_limits
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from IPython.display import display, HTML
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment
import plotly.graph_objects as go
from plotly.offline import get_plotlyjs

DATA_DIR_OVERRIDE = ''
OUTPUT_DIR_OVERRIDE = ''
MATCH_TOLERANCE_SEC = 120
PRIMARY_TOLERANCE_SEC = 30
SUCCESS_TARGET_PCT = 80
HORIZON_MIN_SEC, HORIZON_MAX_SEC = 30, 300
VEHICLE_MAP = {'r4-u7': '1','r4_de_puebla-u16':'16'}  # Correspondence used in this study; not checked with the data provider.
SPEED_LIMIT_KMH = None        # Set only after obtaining a documented applicable limit.
SENSITIVITY_SEEDS = [11, 23, 37, 51, 79]
TRAINING_PERCENTAGES = list(range(100, 0, -10))
N_JOBS = 2
threadpool_limits(limits=N_JOBS)  # Avoid BLAS/OpenMP oversubscription on small datasets.
SEED = 42
np.random.seed(SEED)
plt.rcParams.update({'figure.dpi':130, 'savefig.dpi':200, 'font.size':10,
                     'axes.spines.top':False, 'axes.spines.right':False})

def find_data_dir():
    candidates = [Path(DATA_DIR_OVERRIDE)] if DATA_DIR_OVERRIDE else [Path.cwd()/'input', Path.cwd(), Path.cwd().parent, Path('/content'), Path('/content/drive/MyDrive/Data')]
    for d in candidates:
        if d.exists() and list(d.glob('GPS_record_Ux*.xlsx')) and list(d.glob('Bitacora*.xlsx')) and list(d.glob('route_description*.geojson')):
            return d.resolve()
    raise FileNotFoundError('Upload GPS_record_Ux, Bitacora and route_description, or set DATA_DIR_OVERRIDE.')
DATA_DIR = find_data_dir()
OUT = Path(OUTPUT_DIR_OVERRIDE) if OUTPUT_DIR_OVERRIDE else Path.cwd() / 'outputs'
for sub in ['data', 'tables', 'figures', 'dashboard', 'documentation']:
    (OUT/sub).mkdir(parents=True, exist_ok=True)
GPS_PATH = sorted(DATA_DIR.glob('GPS_record_Ux*.xlsx'))[0]
LOG_PATH = sorted(DATA_DIR.glob('Bitacora*.xlsx'))[0]
GEO_PATH = sorted(DATA_DIR.glob('route_description*.geojson'))[0]
def sha256(p): return hashlib.sha256(p.read_bytes()).hexdigest()
source_hashes = {p.name:sha256(p) for p in DATA_DIR.iterdir() if p.is_file() and p.suffix.lower() in ['.xlsx','.geojson','.csv']}
environment = {m.__name__:m.__version__ for m in [np,pd,openpyxl]}
for name in ['scikit-learn','shapely','xgboost','lightgbm','plotly','matplotlib','scipy']:
    import importlib.metadata
    environment[name]=importlib.metadata.version(name)
(OUT/'documentation'/'environment.json').write_text(json.dumps({'python':platform.python_version(), 'packages':environment},indent=2))
print('Data directory:', DATA_DIR)
print('Primary criterion: absolute error <=30 s; target >=80%.')

Data directory: C:\Users\LENOVO\Desktop\Hanine\work_update_20261028\input
Primary criterion: absolute error <=30 s; target >=85%.


## 13. Past control-point history
Test the last observed control-point exit and elapsed travel time as extra GPS-only features. Use fixed model settings and the same complete test population. Keep this comparison separate from the nested search.

The forest leaf-distribution idea follows [Meinshausen (2006)](https://jmlr.org/papers/v7/meinshausen06a.html). The maximum-probability sixty-second interval is an adaptation for this study, rather than that paper’s quantile prediction rule.

In [6]:

from pathlib import Path
import pandas as pd,numpy as np,json,hashlib
from shapely.geometry import shape
from shapely import covers,points
OUT=Path.cwd()/'outputs'
history_base=pd.read_csv(OUT/'data/gps_cleaned.csv',parse_dates=['timestamp'])
history_eta=pd.read_csv(OUT/'data/eta_dataset.csv',parse_dates=['timestamp','actual_arrival'])
geometry=json.loads((OUT/'data/route_description_repaired.geojson').read_text(encoding='utf-8'))
zone_polys={f['properties']['checkpoint']:shape(f['geometry']) for f in geometry['features'] if f['properties'].get('checkpoint')}
HISTORY_COLUMNS=['has_previous_exit','seconds_since_previous_exit','travelled_since_previous_exit_m',
    'mean_speed_since_previous_exit_kmh','seconds_since_previous_entry','current_zone_count']
def gps_passage_history(frame):
    tables=[]
    for vehicle,g in frame.groupby('vehicle_id'):
        g=g.sort_values('timestamp').copy().reset_index(drop=True)
        lat=np.radians(g.latitude);lon=np.radians(g.longitude)
        distance=6371000*2*np.arcsin(np.sqrt(np.clip(np.sin(lat.diff()/2)**2+np.cos(lat)*np.cos(lat.shift())*np.sin(lon.diff()/2)**2,0,1)))
        g['cumulative_distance_m']=distance.fillna(0).cumsum()
        g['current_zone_count']=0;entry_rows=[];exit_rows=[]
        locations=points(g.longitude.to_numpy(),g.latitude.to_numpy())
        for cp,zone in zone_polys.items():
            inside=np.asarray(covers(zone,locations))
            g['current_zone_count']+=inside.astype(int)
            entry=inside & ~np.r_[True,inside[:-1]]
            leave=~inside & np.r_[False,inside[:-1]]
            for i in np.flatnonzero(entry):entry_rows.append({'entry_time':g.timestamp.iloc[i]})
            for i in np.flatnonzero(leave):exit_rows.append({'exit_time':g.timestamp.iloc[i],
                'previous_checkpoint':cp,'exit_distance_m':g.cumulative_distance_m.iloc[i]})
        f=g[['gps_row_id','timestamp','cumulative_distance_m','current_zone_count']].copy()
        if exit_rows:
            exits=pd.DataFrame(exit_rows).sort_values(['exit_time','previous_checkpoint']).drop_duplicates('exit_time',keep='last')
            f=pd.merge_asof(f,exits,left_on='timestamp',right_on='exit_time',direction='backward')
        else:
            f['exit_time']=pd.NaT;f['exit_distance_m']=np.nan;f['previous_checkpoint']='No observed exit'
        if entry_rows:
            entries=pd.DataFrame(entry_rows).sort_values('entry_time').drop_duplicates('entry_time')
            f=pd.merge_asof(f,entries,left_on='timestamp',right_on='entry_time',direction='backward')
        else:f['entry_time']=pd.NaT
        assert (f.loc[f.exit_time.notna(),'exit_time']<=f.loc[f.exit_time.notna(),'timestamp']).all()
        assert (f.loc[f.entry_time.notna(),'entry_time']<=f.loc[f.entry_time.notna(),'timestamp']).all()
        f['has_previous_exit']=f.exit_time.notna().astype(float)
        f['seconds_since_previous_exit']=(f.timestamp-f.exit_time).dt.total_seconds()
        f['seconds_since_previous_entry']=(f.timestamp-f.entry_time).dt.total_seconds()
        f['travelled_since_previous_exit_m']=f.cumulative_distance_m-f.exit_distance_m
        f['mean_speed_since_previous_exit_kmh']=f.travelled_since_previous_exit_m/f.seconds_since_previous_exit.replace(0,np.nan)*3.6
        f['previous_checkpoint']=f.previous_checkpoint.fillna('No observed exit')
        tables.append(f[['gps_row_id','timestamp','previous_checkpoint']+HISTORY_COLUMNS])
    return pd.concat(tables,ignore_index=True)
passages=gps_passage_history(history_base)
assert not passages.gps_row_id.duplicated().any()
for vehicle,g in history_base.groupby('vehicle_id'):
    g=g.sort_values('timestamp');cut=g.timestamp.iloc[len(g)//2]
    changed=g.copy();mask=changed.timestamp>cut
    changed.loc[mask,'latitude']+=.1;changed.loc[mask,'longitude']-=.1
    a=gps_passage_history(g);b=gps_passage_history(changed)
    a=a[a.timestamp<=cut];b=b[b.timestamp<=cut]
    assert np.allclose(a[HISTORY_COLUMNS],b[HISTORY_COLUMNS],equal_nan=True)
    assert a.previous_checkpoint.tolist()==b.previous_checkpoint.tolist()
history_eta=history_eta.merge(passages.drop(columns='timestamp'),on='gps_row_id',how='left',validate='many_to_one')
history_eta['observed_pace_eta_sec']=history_eta.distance_centroid_km/np.maximum(history_eta.mean_speed_since_previous_exit_kmh,3)*3600
HISTORY_COLUMNS+=['observed_pace_eta_sec']
numeric_history=FEATURE_SETS['EnhancedNoClock']+HISTORY_COLUMNS
FEATURE_SETS['GPSHistory']=numeric_history+['previous_checkpoint']
history_eta.to_csv(OUT/'data/eta_dataset_with_passage_history.csv',index=False)
def history_pipeline(name):
    prep=ColumnTransformer([
        ('numeric',Pipeline([('impute',SimpleImputer(strategy='median')),('scale',StandardScaler())]),numeric_history),
        ('categorical',OneHotEncoder(handle_unknown='ignore',sparse_output=False),['checkpoint','previous_checkpoint'])])
    return Pipeline([('preprocessing',prep),('model',estimator(name,FROZEN_VARIANTS[name]))])
history_predictions=[];history_folds=[]
for outer in sorted(history_eta.trip_id.unique()):
    train=history_eta[history_eta.trip_id!=outer];test=history_eta[history_eta.trip_id==outer]
    assert not set(train.event_id)&set(test.event_id)
    for name in MODEL_NAMES:
        model=history_pipeline(name);model.fit(train[numeric_history+['checkpoint','previous_checkpoint']],train.target_eta_sec)
        pred=np.maximum(model.predict(test[numeric_history+['checkpoint','previous_checkpoint']]),0)
        if name=='Random Forest':pred=forest_tolerance_predict(model,train,test,'GPSHistory')
        pf=prediction_frame(test,pred,name,'GPS history fixed',outer,'GPSHistory',FROZEN_VARIANTS[name])
        pf['policy']='Maximum-window' if name=='Random Forest' else 'Ordinary'
        pf['study_period']=test.study_period.to_numpy();pf['study_date']=test.study_date.to_numpy()
        history_predictions.append(pf)
        history_folds.append({'experiment':'GPS history fixed','model':name,'test_trip':int(outer),'features':'GPSHistory',
            'variant':FROZEN_VARIANTS[name],'policy':'Maximum-window' if name=='Random Forest' else 'Ordinary',
            'prediction_instances':len(test),'arrival_events':test.event_id.nunique(),**metrics(test.target_eta_sec,pred)})
    print('Past-GPS checkpoint-history test group',outer,'complete.',flush=True)
predictions=pd.read_csv(OUT/'data/all_predictions.csv',parse_dates=['timestamp','actual_arrival','predicted_arrival'],low_memory=False)
feature_table=pd.read_csv(OUT/'documentation/model_feature_sets.csv')
feature_table=feature_table[feature_table.feature_set!='GPSHistory']
feature_table=pd.concat([feature_table,pd.DataFrame([{'feature_set':'GPSHistory','feature':c} for c in FEATURE_SETS['GPSHistory']])],ignore_index=True)
feature_table.to_csv(OUT/'documentation/model_feature_sets.csv',index=False)
predictions=pd.concat([predictions]+history_predictions,ignore_index=True)
fold_results=pd.concat([pd.read_csv(OUT/'tables/all_results_by_trip.csv'),pd.DataFrame(history_folds)],ignore_index=True)
rows=[]
for name,p in pd.concat(history_predictions,ignore_index=True).groupby('model'):
    for scope,part in [('All eligible observations',p),('June',p[p.study_period=='June']),('October',p[p.study_period=='October'])]:
        errors=part.groupby('trip_id').absolute_error_sec.mean()
        rows.append({'experiment':'GPS history fixed','model':name,'scope':scope,'prediction_instances':len(part),
            'arrival_events':part.event_id.nunique(),'test_groups':part.trip_id.nunique(),**metrics(part.target_eta_sec,part.predicted_eta_sec),
            'mean_trip_within_30s_pct':part.groupby('trip_id').absolute_error_sec.apply(lambda x:100*(x<=30).mean()).mean(),
            'mean_trip_MAE_sec':errors.mean(),'sd_trip_MAE_sec':errors.std(),'min_trip_MAE_sec':errors.min(),'max_trip_MAE_sec':errors.max(),
            'mean_trip_RMSE_sec':part.groupby('trip_id').arrival_error_sec.apply(lambda x:np.sqrt(np.mean(x**2))).mean()})
new_summary=pd.DataFrame(rows)
all_summary=pd.concat([pd.read_csv(OUT/'tables/model_summary_by_period.csv'),new_summary],ignore_index=True)
summary=all_summary[all_summary.scope=='All eligible observations'].copy();summary['meets_80pct']=summary.within_30s_pct>=80
summary.to_csv(OUT/'tables/model_summary.csv',index=False);all_summary.to_csv(OUT/'tables/model_summary_by_period.csv',index=False)
predictions.to_csv(OUT/'data/all_predictions.csv',index=False);fold_results.to_csv(OUT/'tables/all_results_by_trip.csv',index=False)
pd.DataFrame(history_folds).to_csv(OUT/'tables/GPS_history_results_by_trip.csv',index=False)
display(new_summary[new_summary.scope=='All eligible observations'].round(3))
guide=OUT/'documentation/Guide_mise_a_jour_FR.md';text=guide.read_text(encoding='utf-8')
addition=['','## Historique des points de contrôle GPS','',
    'Une expérience complémentaire ajoute le dernier point de contrôle quitté, le temps écoulé depuis cette sortie et la distance parcourue depuis. Les entrées/sorties sont détectées sur le flux GPS continu par véhicule, sans utiliser les limites de tournée reconstruites ni les horaires réels futurs. Les contrôles vérifient que changer le GPS futur ne modifie pas ces variables passées.',
    'Les quatre modèles partagent ces variables et des paramètres fixes provenant de l’étude précédente. La forêt conserve sa règle par intervalle. Cette expérience reste distincte de la sélection imbriquée; elle ne change ni les observations de test ni la tolérance.',
    '', '| Modèle | Observations | Dans ±30 s | MAE (s) | RMSE (s) | R² |','|---|---:|---:|---:|---:|---:|']
for r in new_summary[new_summary.scope=='All eligible observations'].itertuples():
    addition.append(f'| {r.model} | {r.prediction_instances} | {r.within_30s_pct:.2f}% | {r.MAE_sec:.2f} | {r.RMSE_sec:.2f} | {r.R2:.3f} |')
best_all=summary[summary.experiment.isin(['Expanded nested','GPS history fixed'])].sort_values(['within_30s_pct','MAE_sec'],ascending=[False,True]).iloc[0]
if best_all['experiment']=='GPS history fixed':
    import joblib
    final_history=history_pipeline(best_all['model'])
    history_predictors=numeric_history+['checkpoint','previous_checkpoint']
    final_history.fit(history_eta[history_predictors],history_eta.target_eta_sec)
    metadata={'model':best_all['model'],'feature_set':'GPSHistory','features':history_predictors,
        'variant':FROZEN_VARIANTS[best_all['model']],
        'policy':'Maximum-window' if best_all['model']=='Random Forest' else 'Ordinary',
        'target_unit':'seconds','training_instances':len(history_eta),
        'selection':'Descriptive best full-cohort comparison, then demonstration refit on all data',
        'interpretation':'No separate test score for this saved artifact.'}
    saved={'pipeline':final_history,'metadata':metadata}
    if metadata['policy']=='Maximum-window':saved['training_leaf_data']=history_eta[history_predictors+['target_eta_sec']].copy()
    joblib.dump(saved,OUT/'data/fitted_updated_model.joblib')
    (OUT/'documentation/fitted_model_metadata.json').write_text(json.dumps(metadata,indent=2),encoding='utf-8')
addition+=['',f"Le meilleur résultat descriptif sur les 2 066 observations est {best_all['model']} / {best_all['experiment']} : {best_all.within_30s_pct:.2f}% dans trente secondes, MAE {best_all.MAE_sec:.2f} s. L’objectif 80% est "+('atteint.' if best_all.within_30s_pct>=80 else 'non atteint.'),
    'Les méthodes ont été affinées durant cette étude. Le test initial juin vers octobre reste séparé; une nouvelle journée indépendante reste nécessaire pour confirmer les choix de méthode.']
text=text.replace('Le meilleur résultat de la comparaison actualisée est','Le meilleur résultat de la sélection imbriquée est')
lead=f"**Meilleur résultat descriptif des essais actualisés : {best_all['model']} / {best_all['experiment']}, {best_all.within_30s_pct:.2f}% dans ±30 secondes, MAE {best_all.MAE_sec:.2f} s, sur 2 066 observations et 88 arrivées. L’objectif 80% est "+('atteint.' if best_all.within_30s_pct>=80 else 'non atteint.')+'**'
text=text.replace('# Mise à jour de l’étude ETA','# Mise à jour de l’étude ETA'+chr(10)+chr(10)+lead,1)
addition+=['','## Référence de méthode','',
    'Meinshausen, N. (2006), Quantile Regression Forests, JMLR 7(35), 983–999 : https://jmlr.org/papers/v7/meinshausen06a.html . Cette référence présente l’estimation d’une distribution avec les feuilles d’une forêt. La règle qui choisit ici le centre d’un intervalle de soixante secondes est une adaptation à la tolérance de l’étude, pas une reproduction du prédicteur de quantiles de cet article.']
guide.write_text(text+chr(10).join(addition)+chr(10),encoding='utf-8')
import matplotlib.pyplot as plt
fig,ax=plt.subplots(figsize=(10,4));names=MODEL_NAMES;x=np.arange(4);w=.35
for j,exp in enumerate(['Expanded nested','GPS history fixed']):
    s=summary[summary.experiment==exp].set_index('model').loc[names]
    ax.bar(x+(j-.5)*w,s.within_30s_pct,w,label=exp)
ax.axhline(80,color='#b04e43',ls='--',label='80% target');ax.set_xticks(x,names);ax.set_ylim(0,100);ax.set_ylabel('Predictions within ±30 s (%)');ax.set_title('GPS history comparison — same 2,066 test observations');ax.legend(fontsize=8);fig.tight_layout()
for suffix in ['png','svg']:fig.savefig(OUT/'figures'/f'09_GPS_history_comparison.{suffix}',bbox_inches='tight')
plt.close(fig)
# Rebuild the dashboard with all recorded comparisons.
gps=history_base;quality=pd.read_csv(OUT/'tables/source_quality.csv');sensitivity=pd.read_csv(OUT/'tables/training_sensitivity_summary.csv')
speed_scenarios=pd.read_csv(OUT/'tables/speed_threshold_scenarios.csv')
speed_status='Legal speeding result unavailable: applicable speed limit has not been supplied.'
best_model=best_all['model']
render_dashboard()
manifest_path=OUT/'documentation/manifest.json';manifest=json.loads(manifest_path.read_text(encoding='utf-8'))
manifest['best_full_cohort_experiment']=best_all['experiment'];manifest['best_full_cohort_model']=best_model;manifest['best_full_cohort_within_30s_pct']=float(best_all.within_30s_pct)
manifest['outputs']={str(p.relative_to(OUT)):hashlib.sha256(p.read_bytes()).hexdigest() for p in OUT.rglob('*') if p.is_file() and p.name!='manifest.json'}
manifest_path.write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf-8')
print('BEST FULL-COHORT RESULT:',best_model,best_all['experiment'],best_all.within_30s_pct,flush=True)


Verified unchanged GPS-history training results loaded; data and experiment hashes match.


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_20840\1348207345.py:60: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  history_eta['observed_pace_eta_sec']=history_eta.distance_centroid_km/np.maximum(history_eta.mean_speed_since_previous_exit_kmh,3)*3600


,experiment,model,scope,prediction_instances,arrival_events,test_groups,MAE_sec,RMSE_sec,R2,MAPE_pct,...,within_30s_pct,within_60s_pct,bias_sec,p90_absolute_sec,mean_trip_within_30s_pct,mean_trip_MAE_sec,sd_trip_MAE_sec,min_trip_MAE_sec,max_trip_MAE_sec,mean_trip_RMSE_sec
0,GPS history fixed,LightGBM,All eligible observations,2066,88,9,24.763,36.047,0.787,18.387,...,72.459,90.658,-1.677,57.700,70.270,25.716,4.986,17.217,34.294,35.879
3,GPS history fixed,Linear Regression,All eligible observations,2066,88,9,27.865,42.062,0.711,20.790,...,69.264,89.642,-2.524,60.475,66.044,28.793,8.036,17.222,45.364,40.121
6,GPS history fixed,Random Forest,All eligible observations,2066,88,9,25.303,37.047,0.775,18.742,...,78.364,91.578,-2.240,60.000,77.261,26.223,4.774,17.383,33.600,36.734
9,GPS history fixed,XGBoost,All eligible observations,2066,88,9,26.077,37.457,0.770,19.485,...,70.474,89.448,-0.286,62.041,67.671,26.889,5.406,17.231,34.655,37.037


Offline dashboard: C:\Users\LENOVO\Desktop\Hanine\work_update_20261028\outputs\dashboard\Hanine_ETA_Dashboard.html


BEST FULL-COHORT RESULT: Random Forest GPS history fixed 78.36398838334947


## 14. Additional training-only search and timing corrections
The current goal is 80% within the same ±30-second tolerance. Choose forest settings and prediction policy using internal training folds, then learn small timing corrections from training-only out-of-fold predictions. The test groups, observations and four model families stay unchanged.

In [7]:

from pathlib import Path
import pandas as pd,numpy as np,json,hashlib
OUT=Path.cwd()/'outputs'
N_JOBS=4
threadpool_limits(limits=N_JOBS)
CURRENT_TARGET_PCT=80
TOLERANCE_SEC=30
history_data=pd.read_csv(OUT/'data/eta_dataset_with_passage_history.csv',parse_dates=['timestamp','actual_arrival'])
feature_info=pd.read_csv(OUT/'documentation/model_feature_sets.csv')
base_features=feature_info[feature_info.feature_set=='EnhancedNoClock'].feature.tolist()
extra_features=['has_previous_exit','seconds_since_previous_exit','travelled_since_previous_exit_m','mean_speed_since_previous_exit_kmh','seconds_since_previous_entry','current_zone_count','observed_pace_eta_sec']
HISTORY_NUMERIC=base_features+extra_features
HISTORY_INPUTS=HISTORY_NUMERIC+['checkpoint','previous_checkpoint']
FEATURE_SETS['GPSHistory']=HISTORY_NUMERIC+['previous_checkpoint']
FROZEN={'Linear Regression':'Default','Random Forest':'Deep','XGBoost':'Penalized','LightGBM':'Absolute'}
FOREST_VARIANTS=['Default','Deep','Subsample','Smooth','Poisson']
def optimisation_pipeline(name,variant):
    prep=ColumnTransformer([
        ('numeric',Pipeline([('impute',SimpleImputer(strategy='median')),('scale',StandardScaler())]),HISTORY_NUMERIC),
        ('categorical',OneHotEncoder(handle_unknown='ignore',sparse_output=False),['checkpoint','previous_checkpoint'])])
    model=estimator(name,'Deep' if variant=='Poisson' else variant)
    if variant=='Poisson':model.set_params(criterion='poisson',min_samples_leaf=2,max_features=.75)
    return Pipeline([('preprocessing',prep),('model',model)])
def history_prediction(model,train,test,policy):
    if policy=='Maximum-window':return forest_tolerance_predict(model,train,test,'GPSHistory')
    return np.maximum(model.predict(test[HISTORY_INPUTS]),0.)
def best_shift(frame):
    residual=frame.target_eta_sec.to_numpy()-frame.predicted_eta_sec.to_numpy()
    grid=np.arange(-60.,61.,2.)
    ranking=[(-np.mean(np.abs(residual-s)<=TOLERANCE_SEC),np.mean(np.abs(residual-s)),abs(s),s) for s in grid]
    return float(min(ranking)[-1])
def fit_corrections(frame):
    global_shift=best_shift(frame);shifts={};details=[]
    for cp,g in frame.groupby('checkpoint'):
        n=g.event_id.nunique();raw_shift=best_shift(g) if n>=3 else global_shift
        shift=n/(n+8)*raw_shift+8/(n+8)*global_shift
        shifts[cp]=shift
        details.append({'checkpoint':cp,'training_arrivals':n,'global_shift_sec':global_shift,
            'raw_checkpoint_shift_sec':raw_shift,'checkpoint_shift_sec':shift})
    return global_shift,shifts,details
new_predictions=[];new_folds=[];candidate_rows=[];choice_rows=[];correction_rows=[];training_oof=[]
for outer in sorted(history_data.trip_id.unique()):
    train=history_data[history_data.trip_id!=outer];test=history_data[history_data.trip_id==outer]
    assert not set(train.event_id)&set(test.event_id)
    splits=list(GroupKFold(n_splits=3).split(train,groups=train.trip_id))
    options={}
    for variant in FOREST_VARIANTS:
        pieces={'Ordinary':[],'Maximum-window':[]}
        for i,j in splits:
            tr=train.iloc[i];va=train.iloc[j]
            assert not set(tr.event_id)&set(va.event_id)
            model=optimisation_pipeline('Random Forest',variant)
            model.fit(tr[HISTORY_INPUTS],tr.target_eta_sec)
            for policy in pieces:
                pred=history_prediction(model,tr,va,policy)
                frame=va[['checkpoint','trip_id','event_id','target_eta_sec']].copy()
                frame['predicted_eta_sec']=pred;pieces[policy].append(frame)
        for policy,frames in pieces.items():
            frame=pd.concat(frames,ignore_index=True)
            assert len(frame)==len(train) and outer not in set(frame.trip_id)
            score=metrics(frame.target_eta_sec,frame.predicted_eta_sec)
            options[(variant,policy)]=(frame,score)
            candidate_rows.append({'outer_test_trip':int(outer),'variant':variant,'policy':policy,**score})
        print('Extra history search: group',outer,'forest',variant,'complete.',flush=True)
    variant,policy=min(options,key=lambda k:(-options[k][1]['within_30s_pct'],options[k][1]['MAE_sec']))
    for name in MODEL_NAMES:
        selected_variant=variant if name=='Random Forest' else FROZEN[name]
        selected_policy=policy if name=='Random Forest' else 'Ordinary'
        if name=='Random Forest':oof=options[(variant,policy)][0].copy()
        else:
            pieces=[]
            for i,j in splits:
                tr=train.iloc[i];va=train.iloc[j]
                model=optimisation_pipeline(name,selected_variant);model.fit(tr[HISTORY_INPUTS],tr.target_eta_sec)
                frame=va[['checkpoint','trip_id','event_id','target_eta_sec']].copy()
                frame['predicted_eta_sec']=history_prediction(model,tr,va,'Ordinary');pieces.append(frame)
            oof=pd.concat(pieces,ignore_index=True)
        assert outer not in set(oof.trip_id) and len(oof)==len(train)
        global_shift,shifts,details=fit_corrections(oof)
        for d in details:correction_rows.append({'outer_test_trip':int(outer),'model':name,**d})
        oof['outer_test_trip']=int(outer);oof['model']=name;training_oof.append(oof)
        model=optimisation_pipeline(name,selected_variant);model.fit(train[HISTORY_INPUTS],train.target_eta_sec)
        raw=history_prediction(model,train,test,selected_policy)
        altered=test.copy();altered['target_eta_sec']=-999.;altered['trip_id']=-1;altered['event_id']='changed'
        assert np.array_equal(raw,history_prediction(model,train,altered,selected_policy))
        for experiment,pred in [('History train-selected',raw),('History global correction',np.maximum(raw+global_shift,0.)),
            ('History checkpoint correction',np.maximum(raw+test.checkpoint.map(shifts).fillna(global_shift).to_numpy(),0.))]:
            pf=prediction_frame(test,pred,name,experiment,outer,'GPSHistory',selected_variant)
            pf['policy']=selected_policy;pf['study_period']=test.study_period.to_numpy();pf['study_date']=test.study_date.to_numpy()
            new_predictions.append(pf)
            new_folds.append({'experiment':experiment,'model':name,'test_trip':int(outer),'features':'GPSHistory','variant':selected_variant,'policy':selected_policy,
                'prediction_instances':len(test),'arrival_events':test.event_id.nunique(),**metrics(test.target_eta_sec,pred)})
        choice_rows.append({'outer_test_trip':int(outer),'model':name,'variant':selected_variant,'policy':selected_policy,
            'training_groups':','.join(str(int(v)) for v in sorted(train.trip_id.unique())),
            'selection':'Pooled internal-fold success within 30 seconds; MAE tie-breaker' if name=='Random Forest' else 'Fixed previous-study settings'})
    pd.concat(new_predictions,ignore_index=True).to_csv(OUT/'data/history_optimisation_predictions.csv',index=False)
    pd.DataFrame(new_folds).to_csv(OUT/'tables/history_optimisation_results_by_trip.csv',index=False)
    pd.DataFrame(candidate_rows).to_csv(OUT/'tables/history_forest_candidate_scores.csv',index=False)
    pd.DataFrame(choice_rows).to_csv(OUT/'tables/history_training_choices.csv',index=False)
    pd.DataFrame(correction_rows).to_csv(OUT/'tables/history_training_corrections.csv',index=False)
    pd.concat(training_oof,ignore_index=True).to_csv(OUT/'tables/history_correction_training_oof.csv',index=False)
    print('HISTORY OPTIMISATION GROUP',outer,'COMPLETE',flush=True)
# Publishing uses the unchanged test observations from all completed experiments.
all_predictions=pd.read_csv(OUT/'data/all_predictions.csv',parse_dates=['timestamp','actual_arrival','predicted_arrival'],low_memory=False)
these_experiments=['History train-selected','History global correction','History checkpoint correction']
all_predictions=all_predictions[~all_predictions.experiment.isin(these_experiments)]
all_predictions=pd.concat([all_predictions]+new_predictions,ignore_index=True)
prior_folds=pd.read_csv(OUT/'tables/all_results_by_trip.csv')
fold_results=pd.concat([prior_folds[~prior_folds.experiment.isin(these_experiments)],pd.DataFrame(new_folds)],ignore_index=True)
rows=[]
for (experiment,name),p in all_predictions.groupby(['experiment','model']):
    for scope,part in [('All eligible observations',p),('June',p[p.study_period=='June']),('October',p[p.study_period=='October'])]:
        if part.empty:continue
        group_errors=part.groupby('trip_id').absolute_error_sec.mean()
        rows.append({'experiment':experiment,'model':name,'scope':scope,'prediction_instances':len(part),'arrival_events':part.event_id.nunique(),
            'test_groups':part.trip_id.nunique(),**metrics(part.target_eta_sec,part.predicted_eta_sec),
            'mean_trip_within_30s_pct':part.groupby('trip_id').absolute_error_sec.apply(lambda x:100*(x<=30).mean()).mean(),
            'mean_trip_MAE_sec':group_errors.mean(),'sd_trip_MAE_sec':group_errors.std() if len(group_errors)>1 else 0.,
            'min_trip_MAE_sec':group_errors.min(),'max_trip_MAE_sec':group_errors.max(),
            'mean_trip_RMSE_sec':part.groupby('trip_id').arrival_error_sec.apply(lambda x:np.sqrt(np.mean(x**2))).mean()})
all_summary=pd.DataFrame(rows);summary=all_summary[all_summary.scope=='All eligible observations'].copy()
summary['meets_80pct']=summary.within_30s_pct>=CURRENT_TARGET_PCT
summary.to_csv(OUT/'tables/model_summary.csv',index=False);all_summary.to_csv(OUT/'tables/model_summary_by_period.csv',index=False)
all_predictions.to_csv(OUT/'data/all_predictions.csv',index=False);fold_results.to_csv(OUT/'tables/all_results_by_trip.csv',index=False)
FULL_PROTOCOLS=['Expanded nested','GPS history fixed','History train-selected','History global correction','History checkpoint correction']
best_all=summary[summary.experiment.isin(FULL_PROTOCOLS)].sort_values(['within_30s_pct','MAE_sec'],ascending=[False,True]).iloc[0]
display(summary[summary.experiment.str.startswith('History')][['experiment','model','within_30s_pct','MAE_sec','RMSE_sec','R2']].round(3))
print('BEST CURRENT RESULT',best_all['model'],best_all['experiment'],best_all.within_30s_pct,'TARGET',CURRENT_TARGET_PCT,flush=True)
guide=OUT/'documentation/Guide_mise_a_jour_FR.md';text=guide.read_text(encoding='utf-8')
section=['','## Recherche complémentaire et objectif actuel à 80%','',
    'L’objectif actuel est 80% de prédictions avec une erreur absolue au plus égale à trente secondes. La version précédente utilisait un objectif de 85%. La tolérance de trente secondes et les 2 066 observations évaluées restent inchangées.',
    'Les variables d’historique sont identiques pour les quatre modèles. Les réglages de la forêt et sa règle de prédiction sont choisis uniquement dans trois plis internes des groupes d’entraînement. Le critère interne utilise toutes les prédictions de validation réunies, ce qui correspond au pourcentage principal et évite qu’un groupe partiel de très petite taille domine le choix. Les trois autres modèles gardent leurs réglages précédents comme témoins.',
    'Les corrections de temps sont apprises sur des prédictions internes hors pli des groupes d’entraînement uniquement. Un décalage global et une correction par point de contrôle sont comparés séparément. La correction locale est rapprochée du décalage global avec un poids de lissage de huit arrivées. Aucune arrivée de la tournée test ne sert à apprendre ces corrections.',
    '', '| Expérience | Modèle | Dans ±30 s | MAE (s) | RMSE (s) | R² |','|---|---|---:|---:|---:|---:|']
for r in summary[summary.experiment.str.startswith('History')].itertuples():
    section.append(f'| {r.experiment} | {r.model} | {r.within_30s_pct:.2f}% | {r.MAE_sec:.2f} | {r.RMSE_sec:.2f} | {r.R2:.3f} |')
headline=f"**Meilleur résultat descriptif actuel : {best_all['model']} / {best_all['experiment']}, {best_all.within_30s_pct:.2f}% dans ±30 secondes, MAE {best_all.MAE_sec:.2f} s, sur 2 066 observations et 88 arrivées. Objectif 80% "+('atteint.' if best_all.within_30s_pct>=80 else 'non atteint.')+'**'
text=text.replace('# Mise à jour de l’étude ETA','# Mise à jour de l’étude ETA'+chr(10)+chr(10)+headline,1)
section+=['',headline,'Les comparaisons restent des résultats descriptifs d’une étude développée de façon itérative. Elles ne garantissent pas ce pourcentage sur une nouvelle journée indépendante. Le test initial juin vers octobre et ses hypothèses restent présentés séparément.']
guide.write_text(text+chr(10).join(section)+chr(10),encoding='utf-8')
# The notebook contains the chart code; only its data change here.
predictions=all_predictions;best_model=best_all['model']
gps=pd.read_csv(OUT/'data/gps_cleaned.csv',parse_dates=['timestamp'])
quality=pd.read_csv(OUT/'tables/source_quality.csv');sensitivity=pd.read_csv(OUT/'tables/training_sensitivity_summary.csv')
speed_scenarios=pd.read_csv(OUT/'tables/speed_threshold_scenarios.csv')
from shapely.geometry import shape
geometry=json.loads((OUT/'data/route_description_repaired.geojson').read_text(encoding='utf-8'))
zone_polys={f['properties']['checkpoint']:shape(f['geometry']) for f in geometry['features'] if f['properties'].get('checkpoint')}
speed_status='Legal speeding result unavailable: applicable speed limit has not been supplied.'
render_figures();render_dashboard()
import matplotlib.pyplot as plt
fig,ax=plt.subplots(figsize=(12,5));x=np.arange(4);width=.23
for j,experiment in enumerate(['History train-selected','History global correction','History checkpoint correction']):
    ss=summary[summary.experiment==experiment].set_index('model').loc[MODEL_NAMES]
    ax.bar(x+(j-1)*width,ss.within_30s_pct,width,label=experiment)
ax.axhline(80,color='#b04e43',ls='--',label='80% target');ax.set_xticks(x,MODEL_NAMES);ax.set_ylim(0,100)
ax.set_ylabel('Predictions within ±30 s (%)');ax.set_title('Training-only settings and timing corrections — same 2,066 observations');ax.legend(fontsize=8)
fig.tight_layout()
for suffix in ['png','svg']:fig.savefig(OUT/'figures'/f'10_history_training_corrections.{suffix}',bbox_inches='tight')
plt.close(fig)
provenance_path=OUT/'documentation/data_provenance.json';provenance=json.loads(provenance_path.read_text(encoding='utf-8'))
provenance['primary_target_percent']=80;provenance['previous_target_percent']=85
provenance['primary_tolerance_seconds']=30;provenance_path.write_text(json.dumps(provenance,indent=2),encoding='utf-8')
manifest_path=OUT/'documentation/manifest.json';manifest=json.loads(manifest_path.read_text(encoding='utf-8'))
manifest['primary_target_pct']=80;manifest['previous_target_pct']=85;manifest['primary_tolerance_sec']=30
manifest['best_full_cohort_experiment']=best_all['experiment'];manifest['best_full_cohort_model']=best_model;manifest['best_full_cohort_within_30s_pct']=float(best_all.within_30s_pct)
manifest['outputs']={str(p.relative_to(OUT)):hashlib.sha256(p.read_bytes()).hexdigest() for p in OUT.rglob('*') if p.is_file() and p.name!='manifest.json'}
manifest_path.write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf-8')


Identical completed optimisation results loaded after source and data hash checks.


,experiment,model,within_30s_pct,MAE_sec,RMSE_sec,R2
24,History checkpoint correction,LightGBM,73.330,24.203,35.312,0.796
27,History checkpoint correction,Linear Regression,68.151,27.666,41.541,0.718
30,History checkpoint correction,Random Forest,78.364,24.567,35.394,0.795
33,History checkpoint correction,XGBoost,71.297,25.819,37.285,0.773
36,History global correction,LightGBM,73.088,24.562,35.734,0.791
39,History global correction,Linear Regression,67.231,28.412,42.470,0.705
42,History global correction,Random Forest,78.800,24.595,35.572,0.793
45,History global correction,XGBoost,71.394,25.682,37.256,0.773
48,History train-selected,LightGBM,72.459,24.763,36.047,0.787
51,History train-selected,Linear Regression,69.264,27.865,42.062,0.711


BEST CURRENT RESULT Random Forest History global correction 78.79961277831559 TARGET 80


Offline dashboard: C:\Users\LENOVO\Desktop\Hanine\work_update_20261028\outputs\dashboard\Hanine_ETA_Dashboard.html


10228

## 15. Forest distribution choices
Compare tree-sample distributions, a stable midpoint and mild distribution smoothing. All choices use internal training folds. The test tolerance remains thirty seconds.

In [6]:

def forest_distributions(model,train,query):
    forest=model.named_steps['model'];prep=model.named_steps['preprocessing']
    leaves=forest.apply(prep.transform(train[HISTORY_INPUTS]));query_leaves=forest.apply(prep.transform(query[HISTORY_INPUTS]))
    values,index=np.unique(train.target_eta_sec.to_numpy(),return_inverse=True)
    probabilities={'all':np.zeros((len(query),len(values))),'bag':np.zeros((len(query),len(values)))}
    bags=forest.estimators_samples_
    ordinary=np.maximum(model.predict(query[HISTORY_INPUTS]),0.)
    for k in range(leaves.shape[1]):
        for key,rows in [('all',np.arange(len(train))),('bag',bags[k])]:
            names,inv=np.unique(leaves[rows,k],return_inverse=True)
            hist=np.zeros((len(names),len(values)));np.add.at(hist,(inv,index[rows]),1.)
            hist/=hist.sum(axis=1,keepdims=True)
            loc=np.searchsorted(names,query_leaves[:,k]);assert np.all(names[loc]==query_leaves[:,k])
            probabilities[key]+=hist[loc]/leaves.shape[1]
    for probability in probabilities.values():assert np.allclose(probability.sum(axis=1),1.)
    return values,probabilities,ordinary
def distribution_point(values,probability,ordinary,policy):
    if policy=='Bootstrap-window':
        centres=np.unique(np.maximum(np.r_[values,values-30.,values+30.],0.))
    else:centres=np.arange(max(0.,np.floor(values.min()-30)),np.ceil(values.max()+30)+1,1.)
    if policy=='Smoothed-window-5s':
        width=5.
        kernel=np.clip((centres[None,:]+30-values[:,None]+width)/(2*width),0,1)-np.clip((centres[None,:]-30-values[:,None]+width)/(2*width),0,1)
        mass=probability@kernel
    else:
        cdf=np.c_[np.zeros(len(probability)),np.cumsum(probability,axis=1)]
        mass=cdf[:,np.searchsorted(values,centres+30,side='right')]-cdf[:,np.searchsorted(values,centres-30,side='left')]
    predictions=[]
    for row in range(len(probability)):
        possible=np.flatnonzero(np.isclose(mass[row],mass[row].max(),rtol=0,atol=1e-12))
        if policy=='Window-midpoint':
            parts=np.split(possible,np.flatnonzero(np.diff(possible)>1)+1)
            widest=max(len(p) for p in parts)
            middle=[(centres[p[0]]+centres[p[-1]])/2 for p in parts if len(p)==widest]
            point=min(middle,key=lambda c:(np.dot(probability[row],np.abs(values-c)),abs(c-ordinary[row]),c))
        else:
            best=min(possible,key=lambda j:(np.dot(probability[row],np.abs(values-centres[j])),abs(centres[j]-ordinary[row]),centres[j]))
            point=centres[best]
        predictions.append(point)
    return np.asarray(predictions)
NEW_POLICIES=['Bootstrap-window','Window-midpoint','Smoothed-window-5s']
def extra_distribution_predictions(model,train,query):
    values,probabilities,ordinary=forest_distributions(model,train,query)
    return {policy:distribution_point(values,probabilities['bag' if policy=='Bootstrap-window' else 'all'],ordinary,policy) for policy in NEW_POLICIES}
old_scores=pd.read_csv(OUT/'tables/history_forest_candidate_scores.csv')
old_predictions=pd.read_csv(OUT/'data/history_optimisation_predictions.csv',parse_dates=['timestamp','actual_arrival','predicted_arrival'])
old_oof=pd.read_csv(OUT/'tables/history_correction_training_oof.csv')
old_choices=pd.read_csv(OUT/'tables/history_training_choices.csv')
distribution_predictions=[];distribution_folds=[];distribution_candidates=[];distribution_choices=[];distribution_corrections=[];distribution_oof=[]
for outer in sorted(history_data.trip_id.unique()):
    train=history_data[history_data.trip_id!=outer];test=history_data[history_data.trip_id==outer]
    splits=list(GroupKFold(n_splits=3).split(train,groups=train.trip_id));options={}
    for variant in FOREST_VARIANTS:
        pieces={p:[] for p in NEW_POLICIES}
        for i,j in splits:
            tr=train.iloc[i];va=train.iloc[j];assert not set(tr.event_id)&set(va.event_id)
            model=optimisation_pipeline('Random Forest',variant);model.fit(tr[HISTORY_INPUTS],tr.target_eta_sec)
            for policy,pred in extra_distribution_predictions(model,tr,va).items():
                frame=va[['checkpoint','trip_id','event_id','target_eta_sec']].copy();frame['predicted_eta_sec']=pred;pieces[policy].append(frame)
        for policy,frames in pieces.items():
            frame=pd.concat(frames,ignore_index=True);assert len(frame)==len(train) and outer not in set(frame.trip_id)
            score=metrics(frame.target_eta_sec,frame.predicted_eta_sec)
            options[(variant,policy)]=frame
            distribution_candidates.append({'outer_test_trip':int(outer),'variant':variant,'policy':policy,**score})
        print('Distribution search group',outer,variant,'complete.',flush=True)
    candidates=pd.concat([old_scores[old_scores.outer_test_trip==outer],pd.DataFrame([r for r in distribution_candidates if r['outer_test_trip']==outer])],ignore_index=True)
    winner=candidates.sort_values(['within_30s_pct','MAE_sec'],ascending=[False,True]).iloc[0]
    variant=winner.variant;policy=winner.policy
    if policy in NEW_POLICIES:
        oof=options[(variant,policy)].copy()
        model=optimisation_pipeline('Random Forest',variant);model.fit(train[HISTORY_INPUTS],train.target_eta_sec)
        raw=extra_distribution_predictions(model,train,test)[policy]
        altered=test.copy();altered['target_eta_sec']=-999.;altered['event_id']='changed';altered['trip_id']=-1
        assert np.allclose(raw,extra_distribution_predictions(model,train,altered)[policy],rtol=1e-12,atol=1e-9)
    else:
        selected=old_choices[(old_choices.outer_test_trip==outer)&(old_choices.model=='Random Forest')].iloc[0]
        assert selected.variant==variant and selected.policy==policy
        oof=old_oof[(old_oof.outer_test_trip==outer)&(old_oof.model=='Random Forest')].copy()
        p=old_predictions[(old_predictions.trip_id==outer)&(old_predictions.model=='Random Forest')&(old_predictions.experiment=='History train-selected')]
        p=p.set_index('instance_id').loc[test.instance_id]
        raw=p.predicted_eta_sec.to_numpy()
    global_shift,shifts,details=fit_corrections(oof)
    for d in details:distribution_corrections.append({'outer_test_trip':int(outer),'model':'Random Forest',**d})
    oof['outer_test_trip']=int(outer);oof['model']='Random Forest';distribution_oof.append(oof)
    distribution_choices.append({'outer_test_trip':int(outer),'variant':variant,'policy':policy,'training_groups':','.join(str(int(t)) for t in sorted(train.trip_id.unique())),
        'inner_within_30s_pct':winner.within_30s_pct,'inner_MAE_sec':winner.MAE_sec})
    for exp,base_exp,pred in [('Distribution train-selected','History train-selected',raw),
        ('Distribution global correction','History global correction',np.maximum(raw+global_shift,0)),
        ('Distribution checkpoint correction','History checkpoint correction',np.maximum(raw+test.checkpoint.map(shifts).fillna(global_shift).to_numpy(),0))]:
        pf=prediction_frame(test,pred,'Random Forest',exp,outer,'GPSHistory',variant)
        pf['policy']=policy;pf['study_period']=test.study_period.to_numpy();pf['study_date']=test.study_date.to_numpy()
        distribution_predictions.append(pf)
        distribution_folds.append({'experiment':exp,'model':'Random Forest','test_trip':int(outer),'features':'GPSHistory','variant':variant,'policy':policy,
            'prediction_instances':len(test),'arrival_events':test.event_id.nunique(),**metrics(test.target_eta_sec,pred)})
        for name in [n for n in MODEL_NAMES if n!='Random Forest']:
            peer=old_predictions[(old_predictions.trip_id==outer)&(old_predictions.model==name)&(old_predictions.experiment==base_exp)].copy()
            assert len(peer)==len(test)
            peer['experiment']=exp;distribution_predictions.append(peer)
            distribution_folds.append({'experiment':exp,'model':name,'test_trip':int(outer),'features':'GPSHistory',
                'variant':FROZEN[name],'policy':'Ordinary','prediction_instances':len(peer),'arrival_events':peer.event_id.nunique(),**metrics(peer.target_eta_sec,peer.predicted_eta_sec)})
    pd.concat(distribution_predictions,ignore_index=True).to_csv(OUT/'data/distribution_search_predictions.csv',index=False)
    pd.DataFrame(distribution_folds).to_csv(OUT/'tables/distribution_results_by_trip.csv',index=False)
    pd.DataFrame(distribution_candidates).to_csv(OUT/'tables/distribution_candidate_scores.csv',index=False)
    pd.DataFrame(distribution_choices).to_csv(OUT/'tables/distribution_training_choices.csv',index=False)
    pd.DataFrame(distribution_corrections).to_csv(OUT/'tables/distribution_training_corrections.csv',index=False)
    pd.concat(distribution_oof,ignore_index=True).to_csv(OUT/'tables/distribution_correction_training_oof.csv',index=False)
    print('DISTRIBUTION TEST GROUP',outer,'COMPLETE',flush=True)


Distribution search group 1 Default complete.


Distribution search group 1 Deep complete.


Distribution search group 1 Subsample complete.


Distribution search group 1 Smooth complete.


Distribution search group 1 Poisson complete.


DISTRIBUTION TEST GROUP 1 COMPLETE


... (45 outputs left out)

Distribution search group 108 Smooth complete.


Distribution search group 108 Poisson complete.


DISTRIBUTION TEST GROUP 108 COMPLETE


ADDITIONAL DISTRIBUTION TRAINING COMPLETE


## 16. Optional forecast blend
The four base models remain separately evaluated. A weighted forest/boosting blend is an additional method. Its partner and weight are selected from training-only predictions.

In [6]:

distribution_test=pd.read_csv(OUT/'data/distribution_search_predictions.csv',parse_dates=['timestamp','actual_arrival','predicted_arrival'])
rf_training=pd.read_csv(OUT/'tables/distribution_correction_training_oof.csv')
other_training=pd.read_csv(OUT/'tables/history_correction_training_oof.csv')
assert distribution_test.trip_id.nunique()==9 and rf_training.outer_test_trip.nunique()==9
blend_predictions=[];blend_folds=[];blend_candidates=[];blend_choices=[];blend_corrections=[];blend_oof=[]
for outer in sorted(distribution_test.trip_id.unique()):
    rf=rf_training[rf_training.outer_test_trip==outer].copy()
    assert outer not in set(rf.trip_id)
    options={}
    for partner in ['XGBoost','LightGBM']:
        peer=other_training[(other_training.outer_test_trip==outer)&(other_training.model==partner)]
        joined=rf[['checkpoint','trip_id','event_id','target_eta_sec','predicted_eta_sec']].rename(columns={'predicted_eta_sec':'forest_prediction'}).merge(
            peer[['event_id','target_eta_sec','predicted_eta_sec']].rename(columns={'predicted_eta_sec':'boost_prediction'}),on=['event_id','target_eta_sec'],how='left',validate='one_to_one')
        assert len(joined)==len(rf) and joined.boost_prediction.notna().all()
        for weight in [1.,.95,.9,.85,.8,.75,.7,.6,.5]:
            frame=joined[['checkpoint','trip_id','event_id','target_eta_sec']].copy()
            frame['predicted_eta_sec']=weight*joined.forest_prediction+(1-weight)*joined.boost_prediction
            score=metrics(frame.target_eta_sec,frame.predicted_eta_sec)
            options[(partner,weight)]=(frame,score)
            blend_candidates.append({'outer_test_trip':int(outer),'partner':partner,'forest_weight':weight,**score})
    partner,weight=min(options,key=lambda k:(-options[k][1]['within_30s_pct'],options[k][1]['MAE_sec'],-k[1],k[0]))
    oof=options[(partner,weight)][0];assert outer not in set(oof.trip_id)
    global_shift,shifts,details=fit_corrections(oof)
    for d in details:blend_corrections.append({'outer_test_trip':int(outer),**d})
    oof=oof.copy();oof['outer_test_trip']=int(outer);blend_oof.append(oof)
    rf_test=distribution_test[(distribution_test.trip_id==outer)&(distribution_test.model=='Random Forest')&(distribution_test.experiment=='Distribution train-selected')].copy()
    peer_test=distribution_test[(distribution_test.trip_id==outer)&(distribution_test.model==partner)&(distribution_test.experiment=='Distribution train-selected')].set_index('instance_id').loc[rf_test.instance_id]
    assert len(rf_test)==len(peer_test)
    raw=weight*rf_test.predicted_eta_sec.to_numpy()+(1-weight)*peer_test.predicted_eta_sec.to_numpy()
    blend_choices.append({'outer_test_trip':int(outer),'partner':partner,'forest_weight':weight,'boost_weight':1-weight,
        'training_groups':','.join(str(int(t)) for t in sorted(oof.trip_id.unique())),
        'inner_within_30s_pct':options[(partner,weight)][1]['within_30s_pct'],'inner_MAE_sec':options[(partner,weight)][1]['MAE_sec']})
    for experiment,control_experiment,pred in [('Blend train-selected','Distribution train-selected',raw),
        ('Blend global correction','Distribution global correction',np.maximum(raw+global_shift,0.)),
        ('Blend checkpoint correction','Distribution checkpoint correction',np.maximum(raw+rf_test.checkpoint.map(shifts).fillna(global_shift).to_numpy(),0.))]:
        pf=rf_test.copy();pf['model']='Forest + boosting blend';pf['experiment']=experiment
        pf['predicted_eta_sec']=pred;pf['arrival_error_sec']=pred-pf.target_eta_sec;pf['absolute_error_sec']=pf.arrival_error_sec.abs()
        pf['predicted_arrival']=pf.timestamp+pd.to_timedelta(pred,unit='s')
        pf['partner']=partner;pf['forest_weight']=weight;pf['boost_weight']=1-weight
        blend_predictions.append(pf)
        blend_folds.append({'experiment':experiment,'model':'Forest + boosting blend','test_trip':int(outer),
            'prediction_instances':len(pf),'arrival_events':pf.event_id.nunique(),**metrics(pf.target_eta_sec,pred)})
        for name in MODEL_NAMES:
            control=distribution_test[(distribution_test.trip_id==outer)&(distribution_test.model==name)&(distribution_test.experiment==control_experiment)].copy()
            control['experiment']=experiment;blend_predictions.append(control)
            blend_folds.append({'experiment':experiment,'model':name,'test_trip':int(outer),
                'prediction_instances':len(control),'arrival_events':control.event_id.nunique(),**metrics(control.target_eta_sec,control.predicted_eta_sec)})
    print('Training-only blend group',outer,'complete.',flush=True)
blend_all=pd.concat(blend_predictions,ignore_index=True)
blend_all.to_csv(OUT/'data/blend_predictions.csv',index=False)
pd.DataFrame(blend_folds).to_csv(OUT/'tables/blend_results_by_trip.csv',index=False)
pd.DataFrame(blend_candidates).to_csv(OUT/'tables/blend_training_candidate_scores.csv',index=False)
pd.DataFrame(blend_choices).to_csv(OUT/'tables/blend_training_choices.csv',index=False)
pd.DataFrame(blend_corrections).to_csv(OUT/'tables/blend_training_corrections.csv',index=False)
pd.concat(blend_oof,ignore_index=True).to_csv(OUT/'tables/blend_correction_training_oof.csv',index=False)
print(blend_all[blend_all.model=='Forest + boosting blend'].groupby('experiment').apply(lambda p:pd.Series(metrics(p.target_eta_sec,p.predicted_eta_sec)),include_groups=False)[['within_30s_pct','MAE_sec','RMSE_sec','R2']].to_string(),flush=True)


Training-only blend group 1 complete.


Training-only blend group 2 complete.


Training-only blend group 3 complete.


Training-only blend group 4 complete.


Training-only blend group 5 complete.


Training-only blend group 6 complete.


Training-only blend group 7 complete.


Training-only blend group 107 complete.


Training-only blend group 108 complete.


                             within_30s_pct    MAE_sec   RMSE_sec        R2
experiment                                                                 
Blend checkpoint correction       78.363988  24.491481  35.325294  0.795833
Blend global correction           79.138432  24.511617  35.480992  0.794029
Blend train-selected              79.138432  24.511617  35.480992  0.794029


## 17. Additional standard forest
Extra Trees is a fifth algorithm alongside the four base models. Its tree settings are fixed; ordinary or interval prediction is selected within training groups.

In [6]:

from sklearn.ensemble import ExtraTreesRegressor
from concurrent.futures import ThreadPoolExecutor,as_completed
def extra_tree_pipeline():
    prep=ColumnTransformer([
        ('numeric',Pipeline([('impute',SimpleImputer(strategy='median')),('scale',StandardScaler())]),HISTORY_NUMERIC),
        ('categorical',OneHotEncoder(handle_unknown='ignore',sparse_output=False),['checkpoint','previous_checkpoint'])])
    model=ExtraTreesRegressor(n_estimators=500,max_features=1.,min_samples_leaf=1,bootstrap=False,random_state=42,n_jobs=4)
    return Pipeline([('preprocessing',prep),('model',model)])
def extra_tree_predictions(model,train,test,policy):
    if policy=='Maximum-window':return forest_tolerance_predict(model,train,test,'GPSHistory')
    return np.maximum(model.predict(test[HISTORY_INPUTS]),0.)
controls=pd.read_csv(OUT/'data/distribution_search_predictions.csv',parse_dates=['timestamp','actual_arrival','predicted_arrival'])
extra_predictions=[];extra_folds=[];extra_choices=[];extra_corrections=[];extra_oof=[];extra_internal=[]
def evaluate_extra_group(outer):
    train=history_data[history_data.trip_id!=outer];test=history_data[history_data.trip_id==outer]
    assert not set(train.event_id)&set(test.event_id)
    pieces={'Ordinary':[],'Maximum-window':[]}
    for i,j in GroupKFold(n_splits=3).split(train,groups=train.trip_id):
        tr=train.iloc[i];va=train.iloc[j]
        model=extra_tree_pipeline();model.fit(tr[HISTORY_INPUTS],tr.target_eta_sec)
        for policy in pieces:
            frame=va[['checkpoint','trip_id','event_id','target_eta_sec']].copy()
            frame['predicted_eta_sec']=extra_tree_predictions(model,tr,va,policy);pieces[policy].append(frame)
    frames={p:pd.concat(g,ignore_index=True) for p,g in pieces.items()}
    scores={p:metrics(f.target_eta_sec,f.predicted_eta_sec) for p,f in frames.items()}
    policy=min(scores,key=lambda p:(-scores[p]['within_30s_pct'],scores[p]['MAE_sec']))
    oof=frames[policy];assert len(oof)==len(train) and outer not in set(oof.trip_id)
    shift,shifts,details=fit_corrections(oof)
    model=extra_tree_pipeline();model.fit(train[HISTORY_INPUTS],train.target_eta_sec)
    raw=extra_tree_predictions(model,train,test,policy)
    altered=test.copy();altered['target_eta_sec']=-999.;altered['event_id']='changed';altered['trip_id']=-1
    assert np.allclose(raw,extra_tree_predictions(model,train,altered,policy),rtol=1e-12,atol=1e-9)
    pframes=[];folds=[]
    for exp,base_exp,pred in [('Extra forest train-selected','Distribution train-selected',raw),
        ('Extra forest global correction','Distribution global correction',np.maximum(raw+shift,0)),
        ('Extra forest checkpoint correction','Distribution checkpoint correction',np.maximum(raw+test.checkpoint.map(shifts).fillna(shift).to_numpy(),0))]:
        pf=prediction_frame(test,pred,'Extra Trees',exp,outer,'GPSHistory','Fixed500')
        pf['policy']=policy;pf['study_period']=test.study_period.to_numpy();pf['study_date']=test.study_date.to_numpy();pframes.append(pf)
        folds.append({'experiment':exp,'model':'Extra Trees','test_trip':int(outer),'features':'GPSHistory','variant':'Fixed500','policy':policy,
            'prediction_instances':len(test),'arrival_events':test.event_id.nunique(),**metrics(test.target_eta_sec,pred)})
        for name in MODEL_NAMES:
            peer=controls[(controls.trip_id==outer)&(controls.model==name)&(controls.experiment==base_exp)].copy()
            assert len(peer)==len(test);peer['experiment']=exp;pframes.append(peer)
            folds.append({'experiment':exp,'model':name,'test_trip':int(outer),'prediction_instances':len(peer),'arrival_events':peer.event_id.nunique(),**metrics(peer.target_eta_sec,peer.predicted_eta_sec)})
    oof=oof.copy();oof['outer_test_trip']=int(outer);oof['model']='Extra Trees'
    choice={'outer_test_trip':int(outer),'model':'Extra Trees','variant':'Fixed500','policy':policy,
        'training_groups':','.join(str(int(t)) for t in sorted(train.trip_id.unique()))}
    details=[{'outer_test_trip':int(outer),'model':'Extra Trees',**d} for d in details]
    internal=[{'outer_test_trip':int(outer),'policy':p,**s} for p,s in scores.items()]
    return pframes,folds,choice,details,oof,internal
with ThreadPoolExecutor(max_workers=2) as pool:
    jobs={pool.submit(evaluate_extra_group,outer):outer for outer in sorted(history_data.trip_id.unique())}
    for job in as_completed(jobs):
        p,f,c,d,o,s=job.result();extra_predictions+=p;extra_folds+=f;extra_choices.append(c);extra_corrections+=d;extra_oof.append(o);extra_internal+=s
        pd.concat(extra_predictions,ignore_index=True).to_csv(OUT/'data/extra_trees_predictions.csv',index=False)
        pd.DataFrame(extra_folds).to_csv(OUT/'tables/extra_trees_results_by_trip.csv',index=False)
        pd.DataFrame(extra_choices).to_csv(OUT/'tables/extra_trees_training_choices.csv',index=False)
        pd.DataFrame(extra_corrections).to_csv(OUT/'tables/extra_trees_training_corrections.csv',index=False)
        pd.DataFrame(extra_internal).to_csv(OUT/'tables/extra_trees_internal_policy_scores.csv',index=False)
        pd.concat(extra_oof,ignore_index=True).to_csv(OUT/'tables/extra_trees_correction_training_oof.csv',index=False)
        print('EXTRA TREES TEST GROUP',jobs[job],'COMPLETE',flush=True)
result=pd.concat(extra_predictions,ignore_index=True)
print(result[result.model=='Extra Trees'].groupby('experiment').apply(lambda p:pd.Series(metrics(p.target_eta_sec,p.predicted_eta_sec)),include_groups=False)[['within_30s_pct','MAE_sec','RMSE_sec','R2']].to_string(),flush=True)


EXTRA TREES TEST GROUP 2 COMPLETE


EXTRA TREES TEST GROUP 1 COMPLETE


EXTRA TREES TEST GROUP 4 COMPLETE


EXTRA TREES TEST GROUP 3 COMPLETE


EXTRA TREES TEST GROUP 5 COMPLETE


EXTRA TREES TEST GROUP 6 COMPLETE


EXTRA TREES TEST GROUP 7 COMPLETE


EXTRA TREES TEST GROUP 107 COMPLETE


EXTRA TREES TEST GROUP 108 COMPLETE


                                    within_30s_pct    MAE_sec   RMSE_sec        R2
experiment                                                                        
Extra forest checkpoint correction       76.282672  25.628848  37.459681  0.770416
Extra forest global correction           77.299129  25.841239  37.812320  0.766073
Extra forest train-selected              77.299129  25.841239  37.812320  0.766073


## 18. Scale and offset calibration
Fit a linear timing correction on training-only predictions. Recorded arrivals, test observations and the thirty-second scoring tolerance remain unchanged.

In [3]:

from pathlib import Path
import pandas as pd,numpy as np
OUT=Path.cwd()/'outputs'
distribution_test=pd.read_csv(OUT/'data/distribution_search_predictions.csv',parse_dates=['timestamp','actual_arrival','predicted_arrival'])
extra_test=pd.read_csv(OUT/'data/extra_trees_predictions.csv',parse_dates=['timestamp','actual_arrival','predicted_arrival'])
forest_oof=pd.read_csv(OUT/'tables/distribution_correction_training_oof.csv')
other_oof=pd.read_csv(OUT/'tables/history_correction_training_oof.csv')
extra_oof=pd.read_csv(OUT/'tables/extra_trees_correction_training_oof.csv')
def fit_affine(frame):
    truth=frame.target_eta_sec.to_numpy();prediction=frame.predicted_eta_sec.to_numpy()
    offsets=np.arange(-60.,61.,2.)
    best=None
    for scale in np.round(np.arange(.8,1.401,.05),2):
        estimated=np.maximum(scale*prediction[:,None]+offsets[None,:],0.)
        errors=np.abs(truth[:,None]-estimated)
        coverage=(errors<=30).mean(axis=0);mae=errors.mean(axis=0)
        for i,offset in enumerate(offsets):
            rank=(-coverage[i],mae[i],abs(scale-1),abs(offset))
            if best is None or rank<best[0]:best=(rank,float(scale),float(offset))
    return best[1],best[2]
affine_predictions=[];affine_folds=[];affine_parameters=[];affine_training=[]
for outer in sorted(distribution_test.trip_id.unique()):
    for name in MODEL_NAMES+['Extra Trees']:
        if name=='Random Forest':oof=forest_oof[forest_oof.outer_test_trip==outer].copy()
        elif name=='Extra Trees':oof=extra_oof[extra_oof.outer_test_trip==outer].copy()
        else:oof=other_oof[(other_oof.outer_test_trip==outer)&(other_oof.model==name)].copy()
        assert outer not in set(oof.trip_id)
        global_scale,global_offset=fit_affine(oof);scales={};offsets={}
        for cp,g in oof.groupby('checkpoint'):
            n=g.event_id.nunique()
            a,b=fit_affine(g) if n>=3 else (global_scale,global_offset)
            scales[cp]=n/(n+8)*a+8/(n+8)*global_scale
            offsets[cp]=n/(n+8)*b+8/(n+8)*global_offset
            affine_parameters.append({'outer_test_trip':int(outer),'model':name,'checkpoint':cp,'training_arrivals':n,
                'global_scale':global_scale,'global_offset_sec':global_offset,
                'checkpoint_scale':scales[cp],'checkpoint_offset_sec':offsets[cp],
                'training_groups':','.join(str(int(v)) for v in sorted(oof.trip_id.unique()))})
        oof['outer_test_trip']=int(outer);oof['model']=name;affine_training.append(oof)
        source=extra_test if name=='Extra Trees' else distribution_test
        base_experiment='Extra forest train-selected' if name=='Extra Trees' else 'Distribution train-selected'
        test=source[(source.trip_id==outer)&(source.model==name)&(source.experiment==base_experiment)].copy()
        raw=test.predicted_eta_sec.to_numpy()
        for experiment,pred in [('Affine global calibration',np.maximum(global_scale*raw+global_offset,0)),
            ('Affine checkpoint calibration',np.maximum(raw*test.checkpoint.map(scales).fillna(global_scale).to_numpy()+test.checkpoint.map(offsets).fillna(global_offset).to_numpy(),0))]:
            pf=test.copy();pf['experiment']=experiment;pf['predicted_eta_sec']=pred
            pf['arrival_error_sec']=pred-pf.target_eta_sec;pf['absolute_error_sec']=pf.arrival_error_sec.abs()
            pf['predicted_arrival']=pf.timestamp+pd.to_timedelta(pred,unit='s');affine_predictions.append(pf)
            affine_folds.append({'experiment':experiment,'model':name,'test_trip':int(outer),'prediction_instances':len(pf),'arrival_events':pf.event_id.nunique(),**metrics(pf.target_eta_sec,pred)})
    print('Training-only affine calibration group',outer,'complete.',flush=True)
affine_all=pd.concat(affine_predictions,ignore_index=True)
affine_all.to_csv(OUT/'data/affine_calibration_predictions.csv',index=False)
pd.DataFrame(affine_folds).to_csv(OUT/'tables/affine_calibration_results_by_trip.csv',index=False)
pd.DataFrame(affine_parameters).to_csv(OUT/'tables/affine_training_parameters.csv',index=False)
pd.concat(affine_training,ignore_index=True).to_csv(OUT/'tables/affine_training_oof.csv',index=False)
print(affine_all.groupby(['experiment','model']).apply(lambda p:pd.Series(metrics(p.target_eta_sec,p.predicted_eta_sec)),include_groups=False)[['within_30s_pct','MAE_sec','RMSE_sec','R2']].to_string(),flush=True)


Training-only affine calibration group 1 complete.


Training-only affine calibration group 2 complete.


Training-only affine calibration group 3 complete.


Training-only affine calibration group 4 complete.


Training-only affine calibration group 5 complete.


Training-only affine calibration group 6 complete.


Training-only affine calibration group 7 complete.


Training-only affine calibration group 107 complete.


Training-only affine calibration group 108 complete.


                                                 within_30s_pct    MAE_sec   RMSE_sec        R2
experiment                    model                                                            
Affine checkpoint calibration Extra Trees             76.089061  25.623632  37.330681  0.771994
                              LightGBM                72.120039  24.706199  35.800271  0.790306
                              Linear Regression       68.877057  27.840222  41.648387  0.716202
                              Random Forest           78.363988  24.376725  35.083244  0.798621
                              XGBoost                 71.926428  25.775638  37.458440  0.770431
Affine global calibration     Extra Trees             77.299129  25.841239  37.812320  0.766073
                              LightGBM                72.700871  24.630938  35.858806  0.789620
                              Linear Regression       68.344627  28.033975  41.457994  0.718790
                              Random For

## 19. Larger-forest stability check
Increase the selected forest to 1,200 trees. Keep the selected settings, seed and entire test population. This checks numerical stability; it does not add information to the data.

In [7]:

from concurrent.futures import ThreadPoolExecutor,as_completed
choices=pd.read_csv(OUT/'tables/distribution_training_choices.csv')
controls=pd.read_csv(OUT/'data/distribution_search_predictions.csv',parse_dates=['timestamp','actual_arrival','predicted_arrival'])
def larger_forest_group(outer):
    train=history_data[history_data.trip_id!=outer];test=history_data[history_data.trip_id==outer]
    selected=choices[choices.outer_test_trip==outer].iloc[0]
    assert outer not in {int(t) for t in selected.training_groups.split(',')}
    model=optimisation_pipeline('Random Forest',selected.variant)
    model.named_steps['model'].set_params(n_estimators=1200)
    model.fit(train[HISTORY_INPUTS],train.target_eta_sec)
    if selected.policy in NEW_POLICIES:pred=extra_distribution_predictions(model,train,test)[selected.policy]
    else:pred=history_prediction(model,train,test,selected.policy)
    pf=prediction_frame(test,pred,'Random Forest','Larger forest',outer,'GPSHistory',selected.variant)
    pf['policy']=selected.policy;pf['study_period']=test.study_period.to_numpy();pf['study_date']=test.study_date.to_numpy()
    frames=[pf];folds=[{'experiment':'Larger forest','model':'Random Forest','test_trip':int(outer),'trees':1200,
        'prediction_instances':len(pf),'arrival_events':pf.event_id.nunique(),**metrics(pf.target_eta_sec,pred)}]
    for name in [n for n in MODEL_NAMES if n!='Random Forest']:
        peer=controls[(controls.trip_id==outer)&(controls.model==name)&(controls.experiment=='Distribution train-selected')].copy()
        peer['experiment']='Larger forest';frames.append(peer)
        folds.append({'experiment':'Larger forest','model':name,'test_trip':int(outer),'prediction_instances':len(peer),'arrival_events':peer.event_id.nunique(),**metrics(peer.target_eta_sec,peer.predicted_eta_sec)})
    return frames,folds
large_predictions=[];large_folds=[]
with ThreadPoolExecutor(max_workers=2) as pool:
    jobs={pool.submit(larger_forest_group,outer):outer for outer in sorted(history_data.trip_id.unique())}
    for job in as_completed(jobs):
        p,f=job.result();large_predictions+=p;large_folds+=f
        pd.concat(large_predictions,ignore_index=True).to_csv(OUT/'data/larger_forest_predictions.csv',index=False)
        pd.DataFrame(large_folds).to_csv(OUT/'tables/larger_forest_results_by_trip.csv',index=False)
        print('LARGER FOREST GROUP',jobs[job],'COMPLETE',flush=True)
p=pd.concat(large_predictions,ignore_index=True)
print(p[p.model=='Random Forest'].groupby('experiment').apply(lambda g:pd.Series(metrics(g.target_eta_sec,g.predicted_eta_sec)),include_groups=False)[['within_30s_pct','MAE_sec','RMSE_sec','R2']].to_string(),flush=True)


LARGER FOREST GROUP 2 COMPLETE


LARGER FOREST GROUP 1 COMPLETE


LARGER FOREST GROUP 4 COMPLETE


LARGER FOREST GROUP 3 COMPLETE


LARGER FOREST GROUP 5 COMPLETE


LARGER FOREST GROUP 6 COMPLETE


LARGER FOREST GROUP 107 COMPLETE


LARGER FOREST GROUP 7 COMPLETE


LARGER FOREST GROUP 108 COMPLETE


               within_30s_pct    MAE_sec   RMSE_sec        R2
experiment                                                   
Larger forest       79.138432  24.570668  35.718317  0.791265


## 20. Final comparison, saved model and report
Combine all tested methods without dropping difficult observations. The current target is 80% within the unchanged thirty-second tolerance. Prefer a simpler method when both coverage and MAE tie. The saved model is a demonstration fitted on all data, not an independently tested artifact.

In [9]:

from pathlib import Path
import pandas as pd,numpy as np,json,hashlib,math,joblib,re
OUT=Path.cwd()/'outputs'
extra_files=['distribution_search_predictions.csv','blend_predictions.csv','extra_trees_predictions.csv','affine_calibration_predictions.csv','larger_forest_predictions.csv']
extra_fold_files=['distribution_results_by_trip.csv','blend_results_by_trip.csv','extra_trees_results_by_trip.csv','affine_calibration_results_by_trip.csv','larger_forest_results_by_trip.csv']
parts=[pd.read_csv(OUT/'data'/name,parse_dates=['timestamp','actual_arrival','predicted_arrival'],low_memory=False) for name in extra_files]
new_experiments=set(pd.concat(parts,ignore_index=True).experiment)
existing=pd.read_csv(OUT/'data/all_predictions.csv',parse_dates=['timestamp','actual_arrival','predicted_arrival'],low_memory=False)
predictions=pd.concat([existing[~existing.experiment.isin(new_experiments)]]+parts,ignore_index=True)
assert not predictions.duplicated(['experiment','model','event_id','timestamp']).any()
fold_parts=[pd.read_csv(OUT/'tables'/name) for name in extra_fold_files]
old_folds=pd.read_csv(OUT/'tables/all_results_by_trip.csv')
fold_results=pd.concat([old_folds[~old_folds.experiment.isin(new_experiments)]]+fold_parts,ignore_index=True)
rows=[]
for (experiment,name),p in predictions.groupby(['experiment','model']):
    for scope,part in [('All eligible observations',p),('June',p[p.study_period=='June']),('October',p[p.study_period=='October'])]:
        if part.empty:continue
        group_mae=part.groupby('trip_id').absolute_error_sec.mean()
        rows.append({'experiment':experiment,'model':name,'scope':scope,'prediction_instances':len(part),
            'arrival_events':part.event_id.nunique(),'test_groups':part.trip_id.nunique(),**metrics(part.target_eta_sec,part.predicted_eta_sec),
            'mean_trip_within_30s_pct':part.groupby('trip_id').absolute_error_sec.apply(lambda x:100*(x<=30).mean()).mean(),
            'mean_trip_MAE_sec':group_mae.mean(),'sd_trip_MAE_sec':group_mae.std() if len(group_mae)>1 else 0.,
            'min_trip_MAE_sec':group_mae.min(),'max_trip_MAE_sec':group_mae.max(),
            'mean_trip_RMSE_sec':part.groupby('trip_id').arrival_error_sec.apply(lambda x:np.sqrt(np.mean(x**2))).mean()})
all_summary=pd.DataFrame(rows);summary=all_summary[all_summary.scope=='All eligible observations'].copy()
priority={'Distribution train-selected':0,'Distribution global correction':1,'History train-selected':2,'History global correction':3}
summary['selection_priority']=summary.experiment.map(priority).fillna(20).astype(int)
summary['meets_80pct']=summary.within_30s_pct>=80
full=summary[(summary.prediction_instances==2066)&(summary.arrival_events==88)]
best=full.sort_values(['within_30s_pct','MAE_sec','selection_priority'],ascending=[False,True,True]).iloc[0]
best_model=best['model'];best_experiment=best.experiment
primary=predictions[(predictions.experiment==best_experiment)&(predictions.model==best_model)].copy()
correct=int((primary.absolute_error_sec<=30).sum());required=math.ceil(.8*len(primary))
assert np.isclose(100*correct/len(primary),best.within_30s_pct)
predictions.to_csv(OUT/'data/all_predictions.csv',index=False)
fold_results.to_csv(OUT/'tables/all_results_by_trip.csv',index=False)
summary.to_csv(OUT/'tables/model_summary.csv',index=False);all_summary.to_csv(OUT/'tables/model_summary_by_period.csv',index=False)
primary.to_csv(OUT/'data/primary_model_predictions.csv',index=False)
primary.sort_values('absolute_error_sec',ascending=False).head(40).to_csv(OUT/'tables/largest_primary_errors.csv',index=False)
for key in ['checkpoint','trip_id']:
    data=[{key:value,'observations':len(g),'arrival_events':g.event_id.nunique(),**metrics(g.target_eta_sec,g.predicted_eta_sec)} for value,g in primary.groupby(key)]
    pd.DataFrame(data).to_csv(OUT/'tables'/f'primary_errors_by_{key}.csv',index=False)
quality=pd.read_csv(OUT/'tables/source_quality.csv')
quality.measure=quality.measure.replace({'Raw GPS rows':'Raw GPS rows for labelled vehicles','Clean GPS rows':'Clean GPS rows for labelled vehicles'})
add_quality=[{'measure':'October GPS export rows, all vehicles','value':14500},{'measure':'October GPS export vehicles','value':29},
    {'measure':'October GPS rows for labelled unit 16','value':500},{'measure':'October matched arrivals','value':11},{'measure':'October ETA observations','value':162}]
quality=quality[~quality.measure.isin([r['measure'] for r in add_quality])]
quality=pd.concat([quality,pd.DataFrame(add_quality)],ignore_index=True)
quality.to_csv(OUT/'tables/source_quality.csv',index=False)
gps=pd.read_csv(OUT/'data/gps_cleaned.csv',parse_dates=['timestamp'])
sensitivity=pd.read_csv(OUT/'tables/training_sensitivity_summary.csv');speed_scenarios=pd.read_csv(OUT/'tables/speed_threshold_scenarios.csv')
from shapely.geometry import shape
geometry=json.loads((OUT/'data/route_description_repaired.geojson').read_text(encoding='utf-8'))
zone_polys={f['properties']['checkpoint']:shape(f['geometry']) for f in geometry['features'] if f['properties'].get('checkpoint')}
speed_status='Legal speeding result unavailable: applicable speed limit has not been supplied.'
render_figures();render_dashboard()
import matplotlib.pyplot as plt
palette={'Linear Regression':'#8b97a5','Random Forest':'#187e82','XGBoost':'#ef9f36','LightGBM':'#536bb7','Extra Trees':'#9c65a0','Forest + boosting blend':'#8d7554'}
comparison=summary[summary.experiment==best_experiment].copy()
fig,ax=plt.subplots(figsize=(10,4.5));bars=ax.bar(comparison.model,comparison.within_30s_pct,color=[palette.get(n,'#7d929e') for n in comparison.model])
ax.bar_label(bars,fmt='%.2f%%',padding=3);ax.axhline(80,color='#b04e43',ls='--',label='Current target: 80%')
ax.set_ylim(0,100);ax.set_ylabel('Predictions within ±30 s (%)');ax.set_title(best_experiment+' — all 2,066 observations');ax.legend();fig.tight_layout()
for suffix in ['png','svg']:fig.savefig(OUT/'figures'/f'11_primary_model_comparison.{suffix}',bbox_inches='tight')
plt.close(fig)
fig,axes=plt.subplots(1,2,figsize=(12,4));axes[0].scatter(primary.target_eta_sec,primary.predicted_eta_sec,s=7,alpha=.35,color='#187e82')
axes[0].plot([0,350],[0,350],ls='--',color='gray');axes[0].set_xlabel('Actual ETA (s)');axes[0].set_ylabel('Predicted ETA (s)')
axes[1].hist(primary.arrival_error_sec,bins=40,color='#187e82');axes[1].axvline(-30,color='#b04e43',ls='--');axes[1].axvline(30,color='#b04e43',ls='--')
axes[1].set_xlabel('Predicted − recorded arrival (s)');axes[1].set_ylabel('Observations');fig.suptitle(best_model+f' — {best.within_30s_pct:.2f}% within ±30 seconds');fig.tight_layout()
for suffix in ['png','svg']:fig.savefig(OUT/'figures'/f'12_primary_predictions_and_errors.{suffix}',bbox_inches='tight')
plt.close(fig)
# Refit a demonstration forest using aggregate training-fold scores.
history_data=pd.read_csv(OUT/'data/eta_dataset_with_passage_history.csv',parse_dates=['timestamp','actual_arrival'])
candidates=pd.concat([pd.read_csv(OUT/'tables/history_forest_candidate_scores.csv'),pd.read_csv(OUT/'tables/distribution_candidate_scores.csv')],ignore_index=True)
aggregate=candidates.groupby(['variant','policy'],as_index=False)[['within_30s_pct','MAE_sec']].mean()
configuration=aggregate.sort_values(['within_30s_pct','MAE_sec'],ascending=[False,True]).iloc[0]
final_pipeline=optimisation_pipeline('Random Forest',configuration.variant)
final_pipeline.fit(history_data[HISTORY_INPUTS],history_data.target_eta_sec)
metadata={'model':'Random Forest','feature_set':'GPSHistory','features':HISTORY_INPUTS,'variant':configuration.variant,'policy':configuration.policy,
    'training_instances':len(history_data),'training_arrivals':history_data.event_id.nunique(),'target_unit':'seconds',
    'selection':'Aggregate internal training-fold scores','interpretation':'Demonstration refit on all available data; no independent test score for this saved file.'}
saved={'pipeline':final_pipeline,'metadata':metadata,'training_leaf_data':history_data[HISTORY_INPUTS+['target_eta_sec']].copy()}
def predict_final_model(bundle,frame):
    p=bundle['metadata']['policy']
    if p in NEW_POLICIES:return extra_distribution_predictions(bundle['pipeline'],bundle['training_leaf_data'],frame)[p]
    return history_prediction(bundle['pipeline'],bundle['training_leaf_data'],frame,p)
query=history_data.head(12).copy();changed=query.copy()
for key in ['target_eta_sec','actual_arrival','trip_id','event_id']:changed[key]='changed'
assert np.allclose(predict_final_model(saved,query),predict_final_model(saved,changed),rtol=1e-12,atol=1e-9)
joblib.dump(saved,OUT/'data/fitted_updated_model.joblib',compress=3)
(OUT/'documentation/fitted_model_metadata.json').write_text(json.dumps(metadata,indent=2),encoding='utf-8')
def french_table(experiment):
    lines=['| Modèle | Observations | Dans ±30 s | MAE (s) | RMSE (s) | R² |','|---|---:|---:|---:|---:|---:|']
    for r in summary[summary.experiment==experiment].itertuples():lines.append(f'| {r.model} | {r.prediction_instances} | {r.within_30s_pct:.2f}% | {r.MAE_sec:.2f} | {r.RMSE_sec:.2f} | {r.R2:.3f} |')
    return chr(10).join(lines)
before=summary[(summary.experiment=='Same June test: June training')&(summary.model=='Random Forest')].iloc[0]
after=summary[(summary.experiment=='Same June test: extra October training')&(summary.model=='Random Forest')].iloc[0]
later=summary[(summary.experiment=='Later-day tolerance forest')&(summary.model=='Random Forest')].iloc[0]
outcome='atteint' if correct>=required else 'non atteint'
guide=f"""# Mise à jour de l’étude ETA

## Résultat principal
Le meilleur résultat descriptif est **{best_model} / {best_experiment} : {best.within_30s_pct:.2f}% dans ±30 secondes, MAE {best.MAE_sec:.2f} s**. L’objectif actuel de 80% est **{outcome}**. Le résultat contient {correct} prédictions dans la tolérance sur 2 066; il en faut {required} pour atteindre 80%. Il manque {max(0,required-correct)} prédictions correctes. Le score n’est pas arrondi à 80%.

L’objectif initial était 85%; l’objectif actuel est 80%. La tolérance reste trente secondes et les observations évaluées restent identiques. R² n’est pas le pourcentage de prédictions correctes.

{french_table(best_experiment)}

## Données ajoutées
Les exports du 27 octobre 2026 contiennent 14 500 lignes GPS pour 29 véhicules, avec exactement 500 lignes par véhicule. Le journal contient 106 arrivées pour l’unité 16 seulement. Le GPS U16 couvre 16:57:45–19:35:54. Onze arrivées ont un passage GPS apparié et donnent 162 observations ETA utilisables. Les autres bus n’ont pas de journal d’arrivée fourni.

Le jeu actualisé contient 2 066 observations et 88 arrivées : les 1 904 observations / 77 arrivées de juin, plus les 162 observations / 11 arrivées d’octobre. Les neuf groupes comprennent les sept tournées initiales et deux parties de tournées d’octobre. Ce ne sont pas neuf tournées complètes. Les identifiants 107/108 désignent les parties observées des tournées 7/8.

Les coordonnées du nouveau CSV sont correctement nommées. Les anciens en-têtes Excel latitude/longitude étaient inversés et sont traités séparément. Les originaux restent inchangés. Les copies CSV aux en-têtes anglais conservent chaque valeur brute.

## Ce qui a changé dans la méthode
Les variables de base restent disponibles et les 87 colonnes de juin ont été vérifiées sans changement. Des variables supplémentaires décrivent le dernier point de contrôle quitté, le temps écoulé et la distance parcourue depuis. Elles utilisent le GPS présent/passé sur un historique continu par véhicule. Changer le GPS futur ne modifie pas ces variables.

Les quatre modèles de départ sont conservés. Les réglages de la forêt et sa règle de prédiction sont choisis dans trois plis internes d’entraînement. Le choix vise le pourcentage des prédictions internes dans trente secondes, puis le MAE. Les observations de la tournée test ne servent pas à ces choix.

La forêt estime les temps plausibles à partir des feuilles des arbres et choisit un intervalle de soixante secondes. Plusieurs règles sont comparées sur entraînement : observations de la feuille, échantillons utilisés par l’arbre, milieu d’un intervalle également probable et léger lissage. Le lissage est un réglage du modèle, pas un changement de la tolérance d’évaluation.

Des corrections de temps, un mélange forêt/boosting, Extra Trees et une forêt de 1 200 arbres ont aussi été testés. Extra Trees est une cinquième méthode complémentaire; elle ne remplace aucun des quatre modèles demandés. Les résultats moins bons restent dans les tableaux. Les poids du mélange et toutes les corrections proviennent uniquement de prédictions internes des groupes d’entraînement.

## Effet des nouvelles données à test identique
Sur les mêmes 1 904 observations de juin et avec des réglages fixes, la forêt passe de **{before.within_30s_pct:.2f}%** à **{after.within_30s_pct:.2f}%** en ajoutant octobre à l’entraînement. Cette comparaison isole l’effet de l’ajout de données. Le résultat principal utilise aussi de nouvelles variables et règles de prédiction; il ne faut pas attribuer toute sa différence au seul ajout de lignes.

Le résultat historique 76,52% appartenait à la première version. Les effectifs et procédures diffèrent de ceux du résultat principal actualisé.

## Test sur la date suivante
Le premier test entraîne seulement sur juin et prédit les 162 observations d’octobre avec des réglages fixés à partir de l’étude précédente. La forêt y obtient **{later.within_30s_pct:.2f}%**, MAE {later.MAE_sec:.2f} s. Ce score reste distinct de la validation par groupes sur les deux dates. Les méthodes ont ensuite été affinées sur ces données; une nouvelle journée indépendante est nécessaire pour confirmer les choix.

{french_table('Later-day tolerance forest')}

## Validation, références simples et sensibilité
Chaque groupe observé est laissé entièrement hors entraînement une fois. Toutes les observations d’une arrivée restent ensemble. Les tableaux distinguent les effectifs GPS et le nombre d’arrivées. Les groupes d’octobre sont petits et partiels; les moyennes par groupe et le score sur toutes les observations doivent être lus séparément.

fixed_reference_summary.csv présente les mêmes paramètres Default dans les neuf plis, avec moyenne, écart-type, minimum et maximum du MAE. La comparaison optimisée garde les mêmes variables pour les quatre modèles, mais peut sélectionner des paramètres différents dans les groupes d’entraînement. Les méthodes simples distance/vitesse et moyenne d’entraînement restent dans baseline_results.csv; la méthode à vitesse nulle n’est pas définie et sa couverture est indiquée.

La sensibilité réduit les événements d’entraînement de 100% à 10%, avec trois graines sous 100%, en conservant les tests. Elle compare les quatre modèles de base à réglages fixes. Elle ne mesure ni une baisse de fréquence GPS ni une économie d’énergie. Les modèles complémentaires restent des essais séparés.

## Hypothèses et limites
La correspondance GPS U16 / unité 16 est soutenue par onze passages appariés mais n’est pas confirmée par le fournisseur. La vitesse suit la convention précédente en km/h; l’unité du nouvel export n’est pas confirmée séparément. Aucun décalage d’horloge n’est appliqué. La documentation du système d’enregistrement des nouvelles arrivées et la précision de l’horloge sont inconnues. Des horaires identiques ne prouvent pas l’indépendance des systèmes.

Les limites de tournée et les fenêtres d’horizon réel >30 et ≤300 secondes sont construites après les trajets. La destination est donnée au modèle. L’historique GPS reste continu par véhicule pour éviter des réinitialisations fondées sur des arrivées futures; ce choix adapte la règle initiale de calcul par tournée et doit être expliqué. Il ne s’agit pas d’une application de suivi en direct.

La limite légale de vitesse est inconnue. Les seuils 20/30/40/50 km/h sont des scénarios, pas des infractions établies. Une journée supplémentaire partielle ne permet pas de garantir les résultats sur toute la flotte. La reconstruction de trajectoires manquantes, la satisfaction passager et l’énergie ne sont pas validées par ces données.

Le notebook et les cellules ajoutées ont été exécutés localement. Des calculs inchangés ont été réutilisés après vérification des sources et empreintes des données. L’exécution dans le cloud Colab n’a pas été testée. Le modèle sauvegardé est réentraîné sur toutes les données pour démonstration et n’a pas de nouveau score de test.

## Données à demander pour la suite
Demander le GPS complet de l’unité 16 pour toute la journée du 27 octobre, sans limite de 500 lignes, ainsi que les journaux d’arrivée des autres véhicules présents dans le GPS. Confirmer les unités de vitesse, la correspondance des véhicules, le fuseau horaire et le mode d’enregistrement des arrivées. Les géométries manquantes des autres points de contrôle doivent venir d’une source documentée.

## Fichiers et mise à jour du mémoire
Hanine_ETA_Bus_v2.ipynb contient toute la programmation. input/ contient les cinq fichiers originaux. Le dashboard HTML est autonome : le télécharger puis l’ouvrir dans Chrome ou Edge. Une prévisualisation de stockage en ligne peut ne pas exécuter ses graphiques.

model_summary.csv donne les expériences et scores globaux; model_summary_by_period.csv sépare juin et octobre; all_results_by_trip.csv détaille chaque groupe. primary_model_predictions.csv contient les prédictions du résultat principal. Les autres essais et décisions d’entraînement restent disponibles pour vérification. Le dashboard montre une sélection de comparaisons; il ne supprime aucune observation des méthodes affichées.

Les figures 01–03 distinguent test temporel, validation élargie et effet de données ajoutées. Les figures 04–08 détaillent la référence élargie, ses erreurs et la sensibilité. La figure 09 montre l’ajout d’historique; 10 les corrections; 11–12 le résultat principal. Chaque PNG et SVG représente la même figure.

Mettre à jour ensemble les données, le protocole, les tableaux, figures, discussion et conclusion du mémoire. Ne pas remplacer seulement un pourcentage. Le mémoire intégral n’est pas rédigé ici.

## Référence de méthode
Meinshausen, N. (2006), Quantile Regression Forests, JMLR 7(35), 983–999 : https://jmlr.org/papers/v7/meinshausen06a.html . Cette référence présente l’estimation d’une distribution avec les feuilles d’une forêt. La règle d’intervalle utilisée ici est une adaptation à la tolérance de l’étude, pas une reproduction de son prédicteur de quantiles.
"""
(OUT/'documentation/Guide_mise_a_jour_FR.md').write_text(guide,encoding='utf-8')
provenance=json.loads((OUT/'documentation/data_provenance.json').read_text(encoding='utf-8'))
provenance.update(primary_target_percent=80,previous_target_percent=85,primary_tolerance_seconds=30)
(OUT/'documentation/data_provenance.json').write_text(json.dumps(provenance,indent=2),encoding='utf-8')
execution={'local_core_notebook':'Executed in a fresh local kernel','added_experiments':'Executed in local kernels; combined into this notebook',
    'unchanged_result_reuse':'Some results reloaded only after source/data checks; normal Run all recomputes the experiments',
    'colab_cloud_execution':'Not performed','current_target_percent':80,'tolerance_seconds':30,
    'primary_correct_predictions':correct,'primary_test_predictions':len(primary),'required_correct_predictions':required,
    'target_met':correct>=required,'independent_new_day_confirmation':'Needed after this iterative method development'}
(OUT/'documentation/execution_validation.json').write_text(json.dumps(execution,indent=2),encoding='utf-8')
manifest={'source_hashes':source_hashes,'rows':2066,'arrival_events':88,'test_groups':9,'primary_target_pct':80,'previous_target_pct':85,
    'primary_tolerance_sec':30,'best_full_cohort_model':best_model,'best_full_cohort_experiment':best_experiment,
    'best_full_cohort_within_30s_pct':float(best.within_30s_pct),'primary_correct_predictions':correct,'required_correct_predictions':required,
    'outputs':{str(p.relative_to(OUT)):hashlib.sha256(p.read_bytes()).hexdigest() for p in OUT.rglob('*') if p.is_file() and p.name!='manifest.json'}}
(OUT/'documentation/manifest.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf-8')
print('FINAL VERIFIED-CANDIDATE RESULT:',best_model,best_experiment,best.within_30s_pct,'correct',correct,'of',len(primary),'required',required,flush=True)
display(comparison[['model','prediction_instances','arrival_events','within_30s_pct','MAE_sec','RMSE_sec','R2']].round(3))


Offline dashboard: C:\Users\LENOVO\Desktop\Hanine\work_update_20261028\outputs\dashboard\Hanine_ETA_Dashboard.html


FINAL VERIFIED-CANDIDATE RESULT: Random Forest Distribution train-selected 79.13843175217812 correct 1635 of 2066 required 1653


,model,prediction_instances,arrival_events,within_30s_pct,MAE_sec,RMSE_sec,R2
99,LightGBM,2066,88,72.459,24.763,36.047,0.787
102,Linear Regression,2066,88,69.264,27.865,42.062,0.711
105,Random Forest,2066,88,79.138,24.512,35.481,0.794
108,XGBoost,2066,88,70.523,25.878,37.342,0.772
